# High-Dimensional Regression

**Track:** T2 Supervised Learning · **Difficulty:** advanced · **Prereqs:** [Kernel Regression](/topics/kernel-regression), [Local Polynomial Regression](/topics/local-regression), `concentration-inequalities`, `convex-analysis`, `gradient-descent`

This notebook is the source of truth for the math, code, and figures behind the **High-Dimensional Regression** topic on formalML. Markdown cells carry the exposition (definitions, theorems, proofs); code cells validate the math numerically and produce the figures. The brief at `docs/plans/formalml-high-dimensional-regression-handoff-brief.md` is the implementation spec; this notebook is what gets executed.

Code is CPU-only NumPy / SciPy / scikit-learn. End-to-end runtime target: under 60 seconds on a 2020-era laptop.


In [ ]:
# Preamble — imports, RNG, plot style, output directory.
# formalML notebook conventions (matches kernel-regression, local-regression,
# sparse-bayesian-priors): May-2026 figure-styling baseline; deep-blue / red /
# brown / gray palette; OUT_DIR for figure exports; single seeded RNG threaded
# through the notebook.

from __future__ import annotations
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.linear_model import (
    Lasso, LassoCV, LassoLarsIC, Ridge, ElasticNet, ElasticNetCV,
    LogisticRegression,
)

# --- Reproducibility -------------------------------------------------------
SEED = 42
rng = np.random.default_rng(SEED)

# --- Figure styling — May-2026 formalML baseline ---------------------------
COLOR_OLS       = "#7f7f7f"  # gray  — naive / unregularized baseline
COLOR_RIDGE     = "#7b3c10"  # brown — L2 / classical regularizer
COLOR_LASSO     = "#1f4e79"  # deep blue — central object of this topic
COLOR_ENET      = "#3a6e3a"  # forest — elastic net
COLOR_DEBIASED  = "#c0504d"  # red   — debiased / one-step-corrected estimator
COLOR_TRUTH     = "#000000"  # black — beta* / oracle / target
COLOR_TRAIN     = "#7f7f7f"
COLOR_TEST      = "#1f4e79"

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "figure.figsize": (7.0, 4.0),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlepad": 10,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "cm",
})

# --- Output directory for figures ------------------------------------------
OUT_DIR = Path("figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- DGP-1: the canonical sparse high-dimensional design --------------------
# n = 200, p = 500, s = 10. Rows X_i iid N(0, Sigma) with AR(1) Toeplitz
# Sigma_jk = 0.5^|j-k| (weakly decaying correlation). beta* has the first
# 10 entries equal to 1, the rest 0. Noise sigma = 0.5 => population R^2 ~ 0.99.
# Reused across sections 2-9.

def make_dgp1(n=200, p=500, s=10, sigma=0.5, rho=0.5, rng=None):
    """Generate the canonical sparse high-dimensional DGP for this topic.

    Returns (X, y, beta_star, Sigma).
    """
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(p)
    Sigma = rho ** np.abs(np.subtract.outer(idx, idx))    # AR(1) Toeplitz
    L = np.linalg.cholesky(Sigma)
    Z = rng.standard_normal((n, p))
    X = Z @ L.T
    beta_star = np.zeros(p)
    beta_star[:s] = 1.0
    eps = sigma * rng.standard_normal(n)
    y = X @ beta_star + eps
    return X, y, beta_star, Sigma

# Generate the section 1 sample (referenced as X1, y1, beta_star_1 throughout sections 2-9).
X1, y1, beta_star_1, Sigma_1 = make_dgp1(rng=np.random.default_rng(SEED))

# Sanity: print shape + empirical R^2 (theoretical ~ 0.99 for AR(1) Toeplitz
# with rho=0.5, s=10 active block, sigma=0.5).
signal_var = float(np.var(X1 @ beta_star_1))
noise_var  = 0.5 ** 2
r2_pop = signal_var / (signal_var + noise_var)
print(f"DGP-1: n={X1.shape[0]}, p={X1.shape[1]}, "
      f"s={int((beta_star_1 != 0).sum())}, "
      f"empirical signal-R^2 ~ {r2_pop:.3f}")


## §1. From OLS to penalized regression

When the predictor matrix has more columns than rows, ordinary least squares stops being a single estimator and becomes a degenerate family — infinitely many vectors fit the training data perfectly, none of them generalize. This section establishes the high-dimensional regime, shows OLS failing on the canonical sparse-Gaussian-design problem we'll reuse for the next eight sections, and introduces the two classical regularization remedies: ridge (L2-penalized) and lasso (L1-penalized).

Ridge gives a unique solution and shrinks every coefficient smoothly toward zero. Lasso gives a sparse solution and shrinks small coefficients all the way to zero. The geometric difference between those two penalties — corners on the L1 ball, smooth curvature on the L2 ball — is what makes the lasso the central object of this topic.


### §1.1 The high-dimensional regime $p \gtrsim n$ and where it appears

Standard regression theory assumes more observations than features ($n > p$), often by orders of magnitude. The high-dimensional regime flips that: $p$ is comparable to or much larger than $n$. Three places it shows up routinely:

- **Genome-wide association studies (GWAS):** regress a phenotype on hundreds of thousands to millions of single-nucleotide polymorphisms ($p \approx 10^6$, $n \approx 10^4$).
- **Functional MRI:** predict a behavioral or clinical outcome from $\sim 10^5$ voxel-level activations across $n \sim 10^2$ scans.
- **Text and high-cardinality categorical features:** a bag-of-words encoding pushes $p$ into the millions while $n$ stays in the thousands.

These problems share more than $p \gg n$ — they're also **sparse**: only a small subset of features carries the actual signal. A handful of SNPs drive most of the heritable variation in a quantitative trait; a focal brain region carries most of the predictive signal; a few keywords carry most of the topic information.

We'll formalize sparsity as $\|\boldsymbol\beta^*\|_0 = s$ where $s \ll p$ — the true coefficient vector $\boldsymbol\beta^* \in \mathbb{R}^p$ has only $s$ nonzero entries. The set $S = \{j : \beta^*_j \neq 0\}$ is the **support** and $|S| = s$ is the **sparsity level**. We don't know $S$ in advance; recovering it (or, more modestly, predicting well without recovering it) is the estimator's job.


### §1.2 Why OLS fails: the rank-deficient normal equations

OLS minimizes the squared training loss

$$
\hat{\boldsymbol\beta}^{\text{OLS}} = \arg\min_{\boldsymbol\beta \in \mathbb{R}^p} \frac{1}{n} \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2,
$$

with closed form $\hat{\boldsymbol\beta}^{\text{OLS}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$ when $\mathbf{X}^\top \mathbf{X}$ is invertible. Invertibility requires $\text{rank}(\mathbf{X}) = p$, which in turn requires $n \ge p$.

When $p > n$, $\mathbf{X}^\top \mathbf{X}$ has rank at most $n < p$ and is singular. The normal equations have infinitely many solutions: any vector in the $(p - n)$-dimensional null space of $\mathbf{X}$ can be added to a solution without changing the residual. The OLS objective has a flat plateau of global minima, all interpolating the training data exactly.

The standard pseudoinverse fix picks the minimum-norm solution from that plateau:

$$
\hat{\boldsymbol\beta}^{\text{OLS, min-norm}} = \mathbf{X}^\top (\mathbf{X} \mathbf{X}^\top)^{-1} \mathbf{y}.
$$

But "minimum-norm OLS" is still OLS — it interpolates the training data ($\mathbf{X} \hat{\boldsymbol\beta} = \mathbf{y}$), training MSE is zero, and test MSE is unbounded. Even before $p$ reaches $n$, predictive performance degrades: as $p \to n$ from below, the smallest singular value of $\mathbf{X}$ approaches zero, $(\mathbf{X}^\top \mathbf{X})^{-1}$ blows up, and the OLS coefficient estimates inflate even though the in-sample fit looks great.

**DGP-1 (the §1 toy, reused across §§2-9).** $n = 200$, $p = 500$, $s = 10$. Rows $\mathbf{x}_i \in \mathbb{R}^p$ iid $\mathcal{N}(\mathbf{0}, \boldsymbol\Sigma)$ with $\boldsymbol\Sigma_{jk} = 0.5^{|j-k|}$ (AR(1) Toeplitz). $\beta^*_j = 1$ for $j \in S = \{0, \dots, 9\}$, zero otherwise. Noise $\boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, 0.25\, \mathbf{I})$. Population $R^2 \approx 0.99$ — strong signal, modest noise, sparse truth.


In [ ]:
# §1.2 — OLS overfit demo. Sweep p_used from 10 (the truth) to 199 (just under
# n=200), fit OLS to each subproblem (using only the first p_used columns),
# track train and test MSE. Use a fresh DGP-1 sample (separate from X1) and a
# large test set (n_test = 2000) for stable MSE estimates.

X_tr_f, y_tr_f, beta_star_f, _ = make_dgp1(n=200,  p=500, s=10,
                                            rng=np.random.default_rng(SEED + 100))
X_te_f, y_te_f, _, _           = make_dgp1(n=2000, p=500, s=10,
                                            rng=np.random.default_rng(SEED + 101))

p_grid = [10, 25, 50, 100, 150, 175, 190, 195, 199]
mse_train, mse_test = [], []
for p_used in p_grid:
    X_tr_sub = X_tr_f[:, :p_used]
    X_te_sub = X_te_f[:, :p_used]
    beta_hat, *_ = np.linalg.lstsq(X_tr_sub, y_tr_f, rcond=None)
    mse_train.append(float(np.mean((X_tr_sub @ beta_hat - y_tr_f) ** 2)))
    mse_test .append(float(np.mean((X_te_sub @ beta_hat - y_te_f) ** 2)))
mse_train = np.array(mse_train)
mse_test  = np.array(mse_test)

print(f"{'p_used':>8}  {'train MSE':>10}  {'test MSE':>10}")
for p_used, mtr, mte in zip(p_grid, mse_train, mse_test):
    print(f"{p_used:>8d}  {mtr:>10.4f}  {mte:>10.4f}")

# Sanity assertions on the qualitative shape.
assert np.all(np.diff(mse_train) <= 1e-6), \
    "train MSE should be monotonically nonincreasing in p_used"
assert mse_test[-1] > 5 * mse_test[0], \
    "test MSE should explode by at least 5x as p_used -> n"

# --- Figure 1.1 ---
fig, ax = plt.subplots(figsize=(7.0, 4.5))
ax.plot(p_grid, mse_train, "o-", color=COLOR_TRAIN, label="Train MSE",
        markersize=6, linewidth=1.5)
ax.plot(p_grid, mse_test,  "s-", color=COLOR_TEST,  label="Test MSE",
        markersize=6, linewidth=1.5)
ax.axvline(x=10,  color=COLOR_TRUTH, linestyle=":",  alpha=0.6,
           label=r"True sparsity $s=10$")
ax.axvline(x=200, color="#888888",   linestyle="--", alpha=0.4,
           label=r"$n=200$")
ax.set_yscale("log")
ax.set_xlabel(r"Number of features used in OLS fit, $p_{\mathrm{used}}$")
ax.set_ylabel("MSE (log scale)")
ax.set_title(r"OLS overfit at the high-dim boundary: train MSE $\to 0$, test MSE explodes")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_01_01_ols_overfit.png", bbox_inches="tight")
plt.show()


### §1.3 Ridge regression as the L2 fix

Ridge regression resolves the rank-deficiency by adding a quadratic penalty:

$$
\hat{\boldsymbol\beta}^{\text{ridge}}(\lambda) = \arg\min_{\boldsymbol\beta \in \mathbb{R}^p} \frac{1}{2n} \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2 + \frac{\lambda}{2} \|\boldsymbol\beta\|_2^2,
$$

where $\|\boldsymbol\beta\|_2^2 = \sum_{j=1}^p \beta_j^2$ and $\lambda \ge 0$ is a tuning parameter ($\lambda = 0$ recovers OLS, $\lambda \to \infty$ shrinks every coefficient to zero). The closed form is

$$
\hat{\boldsymbol\beta}^{\text{ridge}}(\lambda) = (\mathbf{X}^\top \mathbf{X} + n \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}.
$$

The matrix $\mathbf{X}^\top \mathbf{X} + n\lambda \mathbf{I}$ is positive definite for any $\lambda > 0$ — eigenvalues bounded below by $n\lambda$ — so the inversion is well-defined whether $p \le n$ or $p > n$. Ridge restores uniqueness.

Two structural properties matter:

1. **Continuous shrinkage, dense solutions.** Each coefficient is shrunk toward zero by a factor that depends on the corresponding singular value of $\mathbf{X}$, but no coefficient is set to exactly zero (with probability one over a continuous design).
2. **Smoothness in $\lambda$.** $\hat{\boldsymbol\beta}^{\text{ridge}}(\lambda)$ is continuous and differentiable in $\lambda$ everywhere on $[0, \infty)$ — coefficients shrink, they don't snap.

Ridge is the right tool when all features carry some signal and the goal is to stabilize coefficient estimates against multicollinearity. In the sparse high-dimensional regime, where the truth has 10 active features out of 500, ridge's refusal to zero out the 490 inactive features is a liability — every irrelevant coefficient contributes variance to the prediction.


In [ ]:
# §1.3 — Ridge on DGP-1 at three penalty levels.
# sklearn's Ridge minimizes (1/2) * ||y - X beta||^2 + alpha * ||beta||^2 (no 1/n out front).
# We use alpha directly as a tuning knob; the absolute scale is unimportant for
# the dense-vs-sparse demonstration.

ridge_alphas = [0.01, 1.0, 100.0]
ridge_fits = []
for alpha in ridge_alphas:
    fit = Ridge(alpha=alpha, fit_intercept=False).fit(X1, y1)
    ridge_fits.append(fit.coef_)

print("Ridge on DGP-1 (n=200, p=500, true sparsity s=10):")
for alpha, beta_hat in zip(ridge_alphas, ridge_fits):
    n_nonzero = int(np.sum(np.abs(beta_hat) > 1e-10))
    print(f"  alpha = {alpha:>7.2f}: {n_nonzero}/{X1.shape[1]} coefficients nonzero, "
          f"||beta||_2 = {np.linalg.norm(beta_hat):.3f}")

# Sanity: ridge should be strictly dense at every alpha > 0.
for beta_hat in ridge_fits:
    assert int(np.sum(np.abs(beta_hat) > 1e-10)) == X1.shape[1], \
        "Ridge solution should be strictly dense (all 500 coefficients nonzero)"


### §1.4 The lasso as the L1 alternative

The lasso (Tibshirani 1996) replaces the squared L2 penalty with an L1 penalty:

$$
\hat{\boldsymbol\beta}^{\text{lasso}}(\lambda) = \arg\min_{\boldsymbol\beta \in \mathbb{R}^p} \frac{1}{2n} \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2 + \lambda \|\boldsymbol\beta\|_1,
$$

where $\|\boldsymbol\beta\|_1 = \sum_{j=1}^p |\beta_j|$. The objective is convex (squared loss is convex; L1 is convex; sum is convex), so any local minimum is global. But the L1 norm is not differentiable at zero, and that non-smoothness has two consequences that turn out to be central:

1. **Sparsity.** The optimal solution has many coefficients exactly equal to zero, with the count controlled by $\lambda$. Geometrically, the L1 ball $\{\boldsymbol\beta : \|\boldsymbol\beta\|_1 \le t\}$ has corners at the coordinate axes; the constrained-form lasso solution is the point where the squared-loss contour first touches the L1 ball as it expands, and that contact point is generically at a corner — i.e., on a coordinate hyperplane, with one or more coefficients exactly zero. The smooth L2 ball has no corners, which is why ridge solutions are dense. We come back to this picture in §2.2.
2. **No closed form in general.** Unlike ridge, the lasso has no closed form for arbitrary $\mathbf{X}$. There is one when $\mathbf{X}^\top \mathbf{X} = n \mathbf{I}$ (orthogonal design — see §3.1), but in general we need an iterative solver. §3 covers coordinate descent, ISTA, and FISTA in detail.

The L1 penalty is the smallest convex penalty that produces sparsity. The non-convex L0 penalty $\|\boldsymbol\beta\|_0 = |\{j : \beta_j \neq 0\}|$ is in some sense the "right" objective for variable selection, but the resulting best-subset problem is NP-hard. The L1 penalty is the convex relaxation of L0 — among convex penalties that produce sparsity, the simplest, the easiest to optimize, and the one with the best-developed statistical theory.


In [ ]:
# §1.4 — Lasso on DGP-1 at three penalty levels: small lambda (under-regularized,
# OLS-like), CV-selected lambda (the "right answer"), and large lambda (over-regularized,
# everything zeroed).

lasso_cv = LassoCV(cv=10, fit_intercept=False, alphas=100,
                   random_state=SEED, max_iter=20_000).fit(X1, y1)
lam_cv = lasso_cv.alpha_

lasso_lams = [0.001, lam_cv, 1.0]
lasso_fits = []
for lam in lasso_lams:
    fit = Lasso(alpha=lam, fit_intercept=False, max_iter=20_000).fit(X1, y1)
    lasso_fits.append(fit.coef_)

print(f"\nLassoCV (10-fold) selected lambda ~ {lam_cv:.4f}\n")
print("Lasso on DGP-1 (n=200, p=500, true sparsity s=10):")
for lam, beta_hat in zip(lasso_lams, lasso_fits):
    n_nonzero = int(np.sum(np.abs(beta_hat) > 1e-10))
    n_in_S    = int(np.sum(np.abs(beta_hat[:10]) > 1e-10))
    print(f"  lambda = {lam:>8.4f}: {n_nonzero:>3d}/{X1.shape[1]} nonzero "
          f"({n_in_S}/10 at true active set S)")

# Sanity: the CV-selected lasso should produce a substantially sparse
# solution that recovers most of the true active set.
beta_cv = lasso_fits[1]
n_nonzero_cv = int(np.sum(np.abs(beta_cv) > 1e-10))
n_in_S_cv    = int(np.sum(np.abs(beta_cv[:10]) > 1e-10))
assert n_nonzero_cv <= 60, \
    f"CV-selected lasso should be substantially sparse ({n_nonzero_cv} > 60)"
assert n_in_S_cv   >= 7, \
    f"CV-selected lasso should recover >=7 of 10 true active coords ({n_in_S_cv} < 7)"

# --- Figures 1.2 & 1.3 — Ridge vs Lasso coefficient bar charts ---
fig, axes = plt.subplots(2, 3, figsize=(11.5, 6.0), sharex=True, sharey=True)

# Top row: ridge.
for ax, alpha, beta_hat in zip(axes[0], ridge_alphas, ridge_fits):
    ax.bar(np.arange(len(beta_hat)), beta_hat, width=1.0,
           color=COLOR_RIDGE, alpha=0.85, label="inactive coords")
    ax.bar(np.arange(10), beta_hat[:10], width=1.0,
           color=COLOR_TRUTH, label=r"true active $j \in S$")
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_title(f"Ridge (alpha = {alpha})")
    if alpha == ridge_alphas[0]:
        ax.set_ylabel(r"$\hat\beta_j$")

# Bottom row: lasso.
for ax, lam, beta_hat in zip(axes[1], lasso_lams, lasso_fits):
    ax.bar(np.arange(len(beta_hat)), beta_hat, width=1.0,
           color=COLOR_LASSO, alpha=0.85)
    ax.bar(np.arange(10), beta_hat[:10], width=1.0, color=COLOR_TRUTH)
    ax.axhline(y=0, color="black", linewidth=0.5)
    cv_tag = " (CV-selected)" if abs(lam - lam_cv) < 1e-6 else ""
    title_lam = f"{lam:.4f}" if lam < 1 else f"{lam:.2f}"
    ax.set_title(f"Lasso (lambda = {title_lam}){cv_tag}")
    ax.set_xlabel(r"Coefficient index $j$")
    if lam == lasso_lams[0]:
        ax.set_ylabel(r"$\hat\beta_j$")

axes[0, 0].set_ylim(-0.6, 1.3)
fig.suptitle(
    "Same data, two penalties. Ridge shrinks every coefficient continuously "
    "(dense). Lasso shrinks small coefficients to exact zero (sparse).\n"
    r"True active coordinates $j \in \{0, \dots, 9\}$ in black.",
    y=1.05, fontsize=11,
)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_01_02_03_ridge_vs_lasso_coefs.png", bbox_inches="tight")
plt.show()


### §1.5 Roadmap

The rest of the topic answers four questions about the lasso.

**What does the estimator look like?** §2 fills in the geometric picture and basic structural results — existence, uniqueness, KKT subgradient conditions.

**How do we compute it?** §3 derives the soft-thresholding closed form for orthogonal designs and develops the iterative solvers (coordinate descent, ISTA, FISTA) used in general.

**Does it predict well, and does it recover the true support?** §§4-6 work out the bias-variance trade-off, prove the headline non-asymptotic prediction-risk bound (the lasso oracle inequality, $O(\sigma^2 s \log p / n)$ under the restricted-eigenvalue condition), and treat variable-selection consistency as a separate theorem with its own sufficient condition (the irrepresentable condition). §7 covers practical $\lambda$ selection by cross-validation and information criteria. §8 covers the ridge / elastic-net / adaptive-lasso variants and when each wins. §9 deepens the geometry of the high-dimensional regime — RIP, sub-Gaussian designs, the implication chain between conditions.

**Can we do inference with it?** §10 is the inferential payoff: naive lasso confidence intervals undercover (PoSI; Berk et al. 2013), and the debiased-lasso construction (Zhang-Zhang 2014; Javanmard-Montanari 2014; van de Geer et al. 2014) restores valid coverage.

§11 extends the lasso to non-Gaussian responses (logistic, Poisson). §12 closes with connections to double/debiased ML, causal inference, and the Bayesian counterpart in [Sparse Bayesian Priors](/topics/sparse-bayesian-priors).


## §2. The lasso estimator

The lasso is convex L1-penalized least squares — a well-defined, well-studied estimator with a clean geometric story and a precise first-order characterization. This section establishes the formal definition (in both penalized and constrained forms), develops the geometric intuition for why L1 penalization produces sparse solutions (corners on the L1 ball, smooth curvature on the L2 ball), addresses existence and uniqueness (Tibshirani 2013), and works out the KKT subgradient conditions.

The KKT conditions are the load-bearing technical machinery for the rest of the topic — the soft-thresholding closed form in §3.1, the basic inequality in the oracle inequality proof of §5.2, and the debiased-lasso construction of §10.2 all derive from them.


### §2.1 The L1-penalized least-squares definition

The lasso estimator is the minimizer of an L1-penalized squared loss:

$$
\hat{\boldsymbol\beta}^{\text{lasso}}(\lambda) = \arg\min_{\boldsymbol\beta \in \mathbb{R}^p} \left\{ \frac{1}{2n} \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2 + \lambda \|\boldsymbol\beta\|_1 \right\}, \qquad \lambda \ge 0.
$$

The factor $1/(2n)$ in front of the squared loss is a convention: the $1/2$ kills a leading 2 in the gradient, and the $1/n$ makes the loss term an empirical average. With this scaling, $\lambda$ has units of "covariate-response correlation" — comparable across sample sizes — and the optimal $\lambda$ scales as $\sigma \sqrt{\log(p) / n}$ (we'll see this in §5). scikit-learn's `Lasso(alpha=λ)` matches the same convention.

The objective is convex (squared loss is convex; L1 is convex; sum of convex is convex), so any local minimum is a global minimum. But the L1 norm is not differentiable at zero — the subgradient of $|\beta_j|$ at zero is the closed interval $[-1, 1]$. That non-smoothness is exactly what produces sparsity, and it's why the lasso has no closed form for general $\mathbf{X}$.

The penalized lasso is equivalent to the **constrained-form** lasso

$$
\hat{\boldsymbol\beta}^{\text{lasso}}(t) = \arg\min_{\|\boldsymbol\beta\|_1 \le t} \frac{1}{2n} \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2,
$$

via Lagrangian duality. For each $\lambda \ge 0$ there's a corresponding budget $t(\lambda) = \|\hat{\boldsymbol\beta}^{\text{lasso}}(\lambda)\|_1$, and conversely each feasible $t$ corresponds to a unique $\lambda(t) \ge 0$. The mapping is monotone but not generally available in closed form. Both forms appear in the literature: the penalized form is more convenient for proofs, the constrained form is more convenient for the geometric pictures we'll draw next.


### §2.2 Geometric picture: $L^1$ corners produce sparsity

Why does L1 penalization produce solutions with exact zeros? The cleanest answer is geometric: the L1 ball has corners on the coordinate axes, and the loss contour generically touches the ball at one of those corners.

Consider the constrained lasso in 2D with a fixed budget $t$. The feasible region $\{\boldsymbol\beta : \|\boldsymbol\beta\|_1 \le t\}$ is a diamond — a square rotated 45 degrees — with vertices at $(\pm t, 0)$ and $(0, \pm t)$. The objective contours $\{\boldsymbol\beta : \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2 = c\}$ are concentric ellipses centered at $\hat{\boldsymbol\beta}^{\text{OLS}}$, with axes determined by the eigenvectors of $\mathbf{X}^\top \mathbf{X}$. The constrained solution is the point where the smallest ellipse just touches the diamond.

If $\hat{\boldsymbol\beta}^{\text{OLS}}$ lies outside the diamond — the interesting case; otherwise the constraint is inactive — the contact point is on the diamond's boundary. **Generically, the contact is at a vertex, not a smooth point on an edge.** At a vertex, one coordinate is exactly zero. Sparsity.

The L2 ball is a smooth disk — no corners. The contact point with an ellipse is generically a smooth point on the disk's boundary, with both coordinates nonzero. Density.

This picture extends to $\mathbb{R}^p$: the L1 ball has $2p$ vertices and a hierarchy of lower-dimensional faces; contact at a $k$-dimensional face means the solution has exactly $p - k$ nonzero coordinates. The L2 ball has no faces of any positive codimension; contact is always smooth, the solution is always dense.

The figure below makes this concrete.


In [ ]:
# §2.2 — Hand-constructed pedagogical figure: L1 ball (diamond) vs L2 ball
# (circle), with elliptical loss contours centered at beta_OLS = (0.4, 1.6).
# At budget t = 1, the L1 contact lands at vertex (0, 1) (sparse); the L2
# contact lands at approximately (0.39, 0.92) (dense).

from scipy.optimize import minimize_scalar

beta_ols_2d = np.array([0.4, 1.6])
H_2d = np.array([[1.0, 0.4], [0.4, 1.0]])
H_inv = np.linalg.inv(H_2d)

def loss2d(B1, B2):
    d1 = B1 - beta_ols_2d[0]
    d2 = B2 - beta_ols_2d[1]
    return 0.5 * (H_2d[0, 0]*d1*d1 + 2*H_2d[0, 1]*d1*d2 + H_2d[1, 1]*d2*d2)

t_l1, t_l2 = 1.0, 1.0

# L1 contact: enumerate the four vertices and four faces (sign quadrants).
candidates = [np.array(v, dtype=float)
              for v in [[t_l1, 0], [-t_l1, 0], [0, t_l1], [0, -t_l1]]]
for sign in [[1, 1], [1, -1], [-1, 1], [-1, -1]]:
    s = np.array(sign, dtype=float)
    denom = float(s @ H_inv @ s)
    lam_face = float((s @ beta_ols_2d - t_l1) / denom)
    if lam_face >= 0:
        beta_face = beta_ols_2d - lam_face * (H_inv @ s)
        if np.all(np.sign(beta_face) * s >= 0):     # face sign condition
            candidates.append(beta_face)
candidates = [c for c in candidates if np.sum(np.abs(c)) <= t_l1 + 1e-9]
beta_l1 = candidates[int(np.argmin([float(loss2d(c[0], c[1])) for c in candidates]))]

# L2 contact: beta = (H + mu*I)^{-1} H beta*, find mu >= 0 with ||beta||_2 = t_l2.
def beta_at_mu(mu):
    return np.linalg.solve(H_2d + mu * np.eye(2), H_2d @ beta_ols_2d)
mu_opt = float(minimize_scalar(
    lambda mu: (np.linalg.norm(beta_at_mu(mu)) - t_l2) ** 2,
    bounds=(1e-6, 100), method="bounded").x)
beta_l2 = beta_at_mu(mu_opt)

print(f"beta_OLS                = {beta_ols_2d}")
print(f"L1 contact (t={t_l1}):  {beta_l1}, ||beta||_1 = {np.sum(np.abs(beta_l1)):.3f}")
print(f"L2 contact (t={t_l2}):  {beta_l2}, ||beta||_2 = {np.linalg.norm(beta_l2):.3f}")

# Sanity: L1 contact at corner (0, 1); L2 contact has both coords nonzero.
assert abs(beta_l1[0]) < 1e-6 and abs(beta_l1[1] - 1.0) < 1e-6, \
    "L1 contact should be at vertex (0, 1) — the sparse-solution corner"
assert np.all(np.abs(beta_l2) > 0.1), \
    "L2 contact should have both coordinates clearly nonzero — dense solution"

# --- Figure 2.1 ---
grid = np.linspace(-0.6, 2.2, 400)
B1_grid, B2_grid = np.meshgrid(grid, grid)
Z = loss2d(B1_grid, B2_grid)

theta = np.linspace(0, 2 * np.pi, 200)
l2_ball = np.stack([t_l2 * np.cos(theta), t_l2 * np.sin(theta)], axis=1)
l1_ball = np.array([[t_l1, 0], [0, t_l1], [-t_l1, 0], [0, -t_l1], [t_l1, 0]],
                   dtype=float)
contour_extra = sorted([0.10, 0.25, 0.6, 1.2, 2.0])

fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.2), sharey=True)
for ax, ball, contact, ball_label, color in [
    (axes[0], l1_ball, beta_l1, r"$L^1$ ball (lasso)",  COLOR_LASSO),
    (axes[1], l2_ball, beta_l2, r"$L^2$ ball (ridge)",  COLOR_RIDGE),
]:
    ax.contour(B1_grid, B2_grid, Z, levels=contour_extra,
               colors="#aaaaaa", linewidths=0.7)
    contact_loss = float(loss2d(contact[0], contact[1]))
    ax.contour(B1_grid, B2_grid, Z, levels=[contact_loss],
               colors="#444444", linewidths=1.3)
    ax.fill(ball[:, 0], ball[:, 1], color=color, alpha=0.12)
    ax.plot(ball[:, 0], ball[:, 1], color=color, linewidth=2.0, label=ball_label)
    ax.scatter(*beta_ols_2d, marker="x", color="black", s=85, zorder=5,
               label=r"$\hat{\boldsymbol{\beta}}^{\mathrm{OLS}}$")
    ax.scatter(*contact, marker="o", color=COLOR_DEBIASED, s=120, zorder=6,
               edgecolors="black", linewidths=1.2,
               label=r"constrained $\hat{\boldsymbol{\beta}}$")
    ax.axhline(0, color="black", linewidth=0.4)
    ax.axvline(0, color="black", linewidth=0.4)
    ax.set_xlim(-0.6, 2.2)
    ax.set_ylim(-0.6, 2.2)
    ax.set_xlabel(r"$\beta_1$")
    ax.set_aspect("equal")
    ax.legend(loc="lower left", fontsize=9)

axes[0].set_ylabel(r"$\beta_2$")
axes[0].set_title(r"$L^1$: contact at vertex $(0, 1)$ $\Rightarrow$ $\hat\beta_1 = 0$ (sparse)")
axes[1].set_title(r"$L^2$: smooth contact $\Rightarrow$ both $\hat\beta_j \neq 0$ (dense)")
fig.suptitle("Why lasso is sparse and ridge is not — the geometric picture",
             y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_02_01_l1_vs_l2_geometry.png", bbox_inches="tight")
plt.show()


### §2.3 Existence and uniqueness

**Existence.** The lasso objective is convex, continuous, and **coercive** — as $\|\boldsymbol\beta\| \to \infty$, the squared loss is bounded below by zero and the L1 penalty grows linearly, so the objective grows without bound. Convex-and-coercive implies attainment of the minimum, so the solution set $\hat{B}(\lambda) = \arg\min_\boldsymbol\beta \{(1/2n) \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|_2^2 + \lambda \|\boldsymbol\beta\|_1\}$ is non-empty for every $\lambda \ge 0$. The solution set is also convex and closed.

**Uniqueness.** When does $\hat{B}(\lambda)$ contain a single point? Two sufficient conditions:

1. **$p \le n$ and $\mathbf{X}$ has full column rank.** Then $\mathbf{X}^\top \mathbf{X}$ is positive definite, the lasso objective is strictly convex, and the minimum is unique.
2. **$\mathbf{X}$ has columns in "general position"** (Tibshirani 2013): no $k+1$ columns of $\pm \mathbf{X}$ lie in an affine subspace of dimension $k - 1$. With probability one over any continuous random design (Gaussian, bounded continuous, etc.), the columns are in general position. So **for every continuous random $\mathbf{X}$, the lasso solution is unique with probability one**, even when $p \gg n$.

The conditions can fail with discrete or rank-deficient designs — duplicate columns, one-hot encodings of categorical features summing to a constant, etc. — in which case $\hat{B}(\lambda)$ is a non-trivial convex set. But **the fitted values $\mathbf{X} \hat{\boldsymbol\beta}$ are always unique**, even when $\hat{\boldsymbol\beta}$ is not. The squared loss is strictly convex in $\mathbf{X} \boldsymbol\beta$, so any two solutions $\hat{\boldsymbol\beta}^{(1)}, \hat{\boldsymbol\beta}^{(2)} \in \hat{B}(\lambda)$ satisfy $\mathbf{X} \hat{\boldsymbol\beta}^{(1)} = \mathbf{X} \hat{\boldsymbol\beta}^{(2)}$. The prediction is uniquely determined; only the coefficient decomposition can be ambiguous.

For DGP-1 the design is continuous Gaussian, so the lasso solution is unique with probability one. We assume uniqueness throughout the rest of the topic.


In [ ]:
# §2.3 — Numerical evidence of uniqueness on DGP-1.
# (1) Run sklearn's Lasso with `selection='random'` (randomized coordinate
#     order) from many seeds. Bit-identical solutions across seeds = strong
#     numerical evidence that the minimum is unique.
# (2) Verify that the active-set columns of X are linearly independent —
#     a Tibshirani-2013 sufficient condition for uniqueness.

print("Uniqueness check on DGP-1 (n=200, p=500, continuous Gaussian X):")

betas = []
for seed in range(20):
    fit = Lasso(alpha=lam_cv, fit_intercept=False, selection="random",
                random_state=seed, max_iter=50_000, tol=1e-12).fit(X1, y1)
    betas.append(fit.coef_)
diffs = [float(np.linalg.norm(betas[0] - b)) for b in betas[1:]]
print(f"  Lasso coordinate-descent randomized over 20 seeds:")
print(f"    max ||beta(seed=0) - beta(seed=k)||_2 = {max(diffs):.2e}  (~ 0 => unique)")
assert max(diffs) < 1e-5, "Lasso solutions should agree across random selection seeds"

# Active set linear independence.
beta_lasso = betas[0]
active_mask = np.abs(beta_lasso) > 1e-10
X_active = X1[:, active_mask]
rank = int(np.linalg.matrix_rank(X_active))
print(f"  Active set size:                   {int(active_mask.sum())}")
print(f"  rank(X_active):                    {rank}")
print(f"  Active columns linearly independent? {rank == int(active_mask.sum())}")
assert rank == int(active_mask.sum()), \
    "Active set columns should be linearly independent (Tibshirani 2013 condition)"


### §2.4 KKT subgradient conditions

The lasso objective is convex but non-differentiable. The first-order optimality condition uses the **subgradient** of the L1 norm in place of the gradient. The subdifferential of $|x|$ is

$$
\partial |x| = \begin{cases} \{\mathrm{sign}(x)\} & x \neq 0, \\ [-1, 1] & x = 0. \end{cases}
$$

The L1 norm $\|\boldsymbol\beta\|_1 = \sum_j |\beta_j|$ has subdifferential $\partial \|\boldsymbol\beta\|_1 = \{\mathbf{g} : g_j \in \partial |\beta_j| \text{ for all } j\}$.

Setting $\boldsymbol{0}$ in the subdifferential of the lasso objective gives the **KKT subgradient conditions**:

> **KKT (lasso).** $\hat{\boldsymbol\beta}$ is a lasso solution at $\lambda > 0$ if and only if there exists $\hat{\mathbf{g}} \in \partial \|\hat{\boldsymbol\beta}\|_1$ such that
> $$
> \frac{1}{n} \mathbf{X}^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta}) = \lambda \, \hat{\mathbf{g}}.
> $$

Coordinate-by-coordinate, two cases:

- **Active ($\hat\beta_j \neq 0$):** $\frac{1}{n} \mathbf{X}_j^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta}) = \lambda \, \mathrm{sign}(\hat\beta_j)$. Residual correlation with each active feature is exactly $\pm \lambda$.
- **Inactive ($\hat\beta_j = 0$):** $\left| \frac{1}{n} \mathbf{X}_j^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta}) \right| \le \lambda$. Residual correlation with each inactive feature is bounded by $\lambda$.

Use $\hat c_j = (1/n) \mathbf{X}_j^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta})$ for the residual-feature correlation. KKT in this notation: $|\hat c_j| = \lambda$ for active $j$, $|\hat c_j| \le \lambda$ for inactive $j$.

**Active set, equicorrelation set.** The **active set** $\hat A_\lambda = \{j : \hat\beta_j \neq 0\}$ is contained in the **equicorrelation set** $\hat E_\lambda = \{j : |\hat c_j| = \lambda\}$. Generically $\hat A_\lambda = \hat E_\lambda$; equality fails only on a measure-zero event for continuous designs.

**Two corollaries.** (i) The active set has size at most $\min(n, p)$ — the lasso never selects more than $n$ features, regardless of $p$. (ii) The KKT conditions provide a **dual certificate** for support recovery: if a subgradient supported on $S$ with $\|\hat{\mathbf{g}}_{S^c}\|_\infty < 1$ satisfies KKT at the oracle restricted estimator, the lasso correctly identifies $S$ as the active set. Constructing this certificate is the irrepresentable condition (§6).


In [ ]:
# §2.4 — Verify KKT subgradient conditions on the §1 lasso solution.
# Active condition:    (1/n) X_j' (y - X beta_hat) = lambda * sign(beta_hat_j)  for j in A_hat
# Inactive condition: |(1/n) X_j' (y - X beta_hat)| <= lambda                    for j not in A_hat

beta_lasso = lasso_fits[1]      # CV-selected fit from §1
n1 = X1.shape[0]
lam = lam_cv

residual = y1 - X1 @ beta_lasso
c_hat = (X1.T @ residual) / n1   # residual-feature correlations

active   = np.abs(beta_lasso) > 1e-10
inactive = ~active

# Active KKT residual: c_hat_j - lambda * sign(beta_hat_j) should be ~ 0.
active_kkt    = c_hat[active] - lam * np.sign(beta_lasso[active])
active_max_err = float(np.max(np.abs(active_kkt)))

# Inactive KKT slack: |c_hat_j| <= lambda.
inactive_max_corr = float(np.max(np.abs(c_hat[inactive])))

print(f"KKT verification on §1 lasso fit (lambda = {lam:.4f}, |A_hat| = {int(active.sum())}):")
print(f"  Active condition:   max |c_hat_j - lambda sign(beta_hat_j)| = {active_max_err:.2e}")
print(f"  Inactive condition: max_{{j not in A_hat}} |c_hat_j|         = {inactive_max_corr:.4f}")
print(f"                      lambda                                    = {lam:.4f}")

assert active_max_err <= 5e-4, \
    "Active KKT condition should hold tightly (residual correlation ~ pm lambda)"
assert inactive_max_corr <= lam + 5e-4, \
    "Inactive KKT condition should hold (residual correlation bounded by lambda)"

# --- Figure 2.2: KKT in pictures ---
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(np.abs(c_hat[inactive]), bins=40, color=COLOR_LASSO, alpha=0.55,
        label=r"$|\hat c_j|$, inactive coords ($\hat\beta_j = 0$)")
# Active coords sit exactly at lambda — show as a rug at y = small offset.
y_rug = ax.get_ylim()[1] * 0.05
ax.scatter(np.abs(c_hat[active]),
           np.full(int(active.sum()), y_rug),
           color=COLOR_TRUTH, s=40, zorder=5, marker="|",
           label=r"$|\hat c_j|$, active coords ($\hat\beta_j \neq 0$)")
ax.axvline(lam, color=COLOR_DEBIASED, linewidth=1.6, linestyle="--",
           label=fr"$\lambda = {lam:.4f}$")
ax.set_xlabel(r"$|\hat c_j| = \left|\frac{1}{n} \mathbf{X}_j^\top (\mathbf{y} - \mathbf{X}\hat{\boldsymbol{\beta}})\right|$")
ax.set_ylabel("count (inactive)")
ax.set_title(r"KKT in pictures: inactive $|\hat c_j| < \lambda$, active $|\hat c_j| = \lambda$")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_02_02_kkt_verification.png", bbox_inches="tight")
plt.show()


## §3. Solving the lasso

The lasso has no closed form for general $\mathbf{X}$, so we need iterative algorithms. This section develops the four solvers that matter in practice, in increasing order of sophistication: the **soft-thresholding closed form** for orthogonal designs (§3.1, the only case that admits a closed form, but conceptually load-bearing because every general-purpose solver reduces to it inside the inner loop); **coordinate descent** (§3.2, the `glmnet` workhorse); **ISTA**, the proximal-gradient method (§3.3, simple and slow, $O(1/k)$ rate); and **FISTA** with Nesterov momentum (§3.4, the same framework with a momentum trick that improves the rate to $O(1/k^2)$). §3.5 gives practical solver-choice notes.

A common thread: every solver is some application of the **soft-thresholding operator** $S(z, \lambda) = \mathrm{sign}(z) \cdot \max(|z| - \lambda, 0)$, which is the proximal operator of the L1 norm. ISTA / FISTA / CD differ only in *what* they soft-threshold and *how often*.


### §3.1 Soft-thresholding closed form for orthogonal designs

When $\mathbf{X}^\top \mathbf{X} = n \mathbf{I}$ — an **orthogonal design**, achievable in practice via QR decomposition of any full-rank $\mathbf{X}$ — the lasso decouples across coordinates and admits a closed-form solution.

Substitute $\mathbf{X}^\top \mathbf{X} = n \mathbf{I}$ into the lasso objective and let $\mathbf{z} = \mathbf{X}^\top \mathbf{y} / n$. The objective separates by coordinate into $p$ independent univariate problems:

$$
\min_{\beta_j} \; \frac{1}{2}(\beta_j - z_j)^2 + \lambda |\beta_j|, \qquad j = 1, \dots, p.
$$

> **Theorem 3.1 (soft-thresholding).** The unique minimizer of $\frac{1}{2}(\beta - z)^2 + \lambda |\beta|$ is
> $$
> S(z, \lambda) := \mathrm{sign}(z) \cdot \max(|z| - \lambda, 0) = \begin{cases} z - \lambda & z > \lambda, \\ 0 & |z| \le \lambda, \\ z + \lambda & z < -\lambda. \end{cases}
> $$

**Proof.** The objective is convex and coercive, so a unique minimum exists. KKT: $0 \in \hat\beta - z + \lambda \, \partial |\hat\beta|$, equivalently $z - \hat\beta \in \lambda \, \partial |\hat\beta|$.

- *$\hat\beta > 0$:* $\partial |\hat\beta| = \{1\}$, so $\hat\beta = z - \lambda$, requiring $z > \lambda$.
- *$\hat\beta < 0$:* $\partial |\hat\beta| = \{-1\}$, so $\hat\beta = z + \lambda$, requiring $z < -\lambda$.
- *$\hat\beta = 0$:* $\partial |\hat\beta| = [-1, 1]$, requiring $|z| \le \lambda$.

These three cases partition $\mathbb{R}$, giving $\hat\beta = S(z, \lambda)$. $\blacksquare$

The geometric content: a "dead zone" $|z_j| \le \lambda$ produces the sparsity, and constant shrinkage $|\hat\beta_j| = |z_j| - \lambda$ outside the dead zone biases active coefficients toward zero (this is the bias problem the debiased lasso fixes in §10).

For non-orthogonal designs no closed form exists. But $S(z, \lambda)$ remains the algorithmic atom: it's the **proximal operator** of the L1 norm, $\mathrm{prox}_{\eta \lambda \|\cdot\|_1}(\mathbf{z}) = S(\mathbf{z}, \eta \lambda)$ applied componentwise. Coordinate descent and proximal-gradient methods both apply $S$ inside their iterations.


In [ ]:
# §3.1 — Soft-thresholding operator and orthogonal-design verification.

def soft_threshold(z, lam):
    """The L1 proximal operator: S(z, lam) = sign(z) * max(|z| - lam, 0).

    Vectorized over arrays. The 3-line algorithmic kernel of every lasso solver.
    """
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

# Verify on an orthogonal design: build X with X^T X = n I via QR factorization,
# fit lasso with sklearn, compare to the closed-form S(X^T y / n, lambda).

n_orth, p_orth = 200, 50
Z = np.random.default_rng(SEED + 200).standard_normal((n_orth, p_orth))
Q, _ = np.linalg.qr(Z)
X_orth = Q * np.sqrt(n_orth)            # ||X_j||^2 = n, X^T X = n*I
gram = X_orth.T @ X_orth / n_orth
assert np.allclose(gram, np.eye(p_orth), atol=1e-10), "X should be orthogonal"

# Sparse signal + Gaussian noise.
beta_true_orth = np.zeros(p_orth)
beta_true_orth[:5] = [2.0, 1.5, 1.0, 0.5, 0.3]
y_orth = (X_orth @ beta_true_orth
          + 0.2 * np.random.default_rng(SEED + 201).standard_normal(n_orth))

# Closed-form lasso solution: beta_j = S(z_j, lambda), z = X^T y / n.
z = X_orth.T @ y_orth / n_orth
lam_orth = 0.1
beta_closed = soft_threshold(z, lam_orth)

# sklearn solution, tight tolerance.
beta_sklearn = Lasso(alpha=lam_orth, fit_intercept=False, tol=1e-12,
                     max_iter=20_000).fit(X_orth, y_orth).coef_

diff = float(np.linalg.norm(beta_closed - beta_sklearn))
print(f"Orthogonal design (n={n_orth}, p={p_orth}, lambda={lam_orth}):")
print(f"  ||beta_closed_form - beta_sklearn||_2 = {diff:.2e}")
print(f"  Active set match? {np.array_equal(np.abs(beta_closed) > 1e-10, np.abs(beta_sklearn) > 1e-10)}")
assert diff < 1e-8, \
    "Closed-form lasso and sklearn should match to ~machine precision on orthogonal designs"


### §3.2 Coordinate descent

Coordinate descent solves the lasso by cycling through coordinates and minimizing the objective over one coordinate at a time. Each subproblem is univariate and admits a closed form via soft-thresholding.

Fix all coordinates except $\beta_j$. Let $\mathbf{r}_{-j} = \mathbf{y} - \sum_{k \neq j} \mathbf{X}_k \beta_k$ be the **partial residual**. The lasso objective restricted to $\beta_j$ is

$$
F_j(\beta_j) = \frac{c_j}{2} \beta_j^2 - z_j \beta_j + \lambda |\beta_j| + \mathrm{const}, \qquad c_j = \tfrac{\|\mathbf{X}_j\|^2}{n}, \quad z_j = \tfrac{\mathbf{X}_j^\top \mathbf{r}_{-j}}{n}.
$$

This is the same univariate problem from §3.1 up to the rescaling $c_j$. The minimizer:

$$
\hat\beta_j^{\text{new}} = \frac{S(z_j, \lambda)}{c_j}.
$$

**Algorithm.** Initialize $\boldsymbol\beta = \mathbf{0}$, $\mathbf{r} = \mathbf{y}$. Cycle over $j = 1, \dots, p$: form the partial residual $\tilde{\mathbf{r}} = \mathbf{r} + \mathbf{X}_j \beta_j$, compute $z_j = \mathbf{X}_j^\top \tilde{\mathbf{r}}/n$, update $\beta_j^{\text{new}} = S(z_j, \lambda)/c_j$, then update the full residual $\mathbf{r} \leftarrow \mathbf{r} - \mathbf{X}_j (\beta_j^{\text{new}} - \beta_j)$. Storing $\mathbf{r}$ and updating it incrementally costs $O(n)$ per coordinate, $O(np)$ per cycle — same as one ISTA / FISTA iteration.

**Convergence.** Tseng (2001) showed coordinate descent converges to a global minimum for any "smooth + separable convex" objective. No global rate guarantee, but linear convergence under additional assumptions, and on continuous-design lasso problems CD is one of the fastest methods in practice — Friedman, Hastie & Tibshirani (2010) report 10–100× speedups over LARS / proximal gradient at typical $(n, p)$ scales. `glmnet` and scikit-learn's `Lasso` both use CD as the default.

**Warm starts along a $\lambda$ path.** Practical solvers compute $\hat{\boldsymbol\beta}(\lambda)$ on a decreasing $\lambda$-grid, using the previous solution to warm-start the next. The path is piecewise linear in $\lambda$ (Efron-Hastie-Johnstone-Tibshirani 2004), so each $\lambda$-step needs only 5–20 CD passes to converge.


### §3.3 ISTA: the proximal gradient method

For more general non-smooth penalties (group lasso, fused lasso, nuclear norm) we need a different framework. **Proximal gradient methods** generalize gradient descent to objectives $F = f + g$ where $f$ is smooth and $g$ is non-smooth but admits a tractable prox. For the lasso, $f(\boldsymbol\beta) = (1/2n) \|\mathbf{y} - \mathbf{X} \boldsymbol\beta\|^2$ and $g(\boldsymbol\beta) = \lambda \|\boldsymbol\beta\|_1$, with $\mathrm{prox}_{\eta g}(\mathbf{z}) = S(\mathbf{z}, \eta \lambda)$.

The **iterative soft-thresholding algorithm (ISTA):**

$$
\boldsymbol\beta^{k+1} = S\!\left( \boldsymbol\beta^k - \frac{\eta}{n} \mathbf{X}^\top (\mathbf{X} \boldsymbol\beta^k - \mathbf{y}), \; \eta \lambda \right),
$$

with step size $\eta = 1/L$, $L = \|\mathbf{X}\|_2^2 / n$ (the Lipschitz constant of $\nabla f$). Each iteration costs one matrix-vector multiply $\mathbf{X} \boldsymbol\beta^k$ and one $\mathbf{X}^\top \mathbf{r}$ — $O(np)$, same as a CD cycle.

> **Theorem 3.2 (ISTA convergence).** With step size $1/L$,
> $$
> F(\boldsymbol\beta^k) - F(\boldsymbol\beta^*) \le \frac{L}{2k} \|\boldsymbol\beta^0 - \boldsymbol\beta^*\|^2 \quad \text{for all } k \ge 1.
> $$

**Proof sketch.** Two ingredients: a per-step descent lemma (Beck-Teboulle 2009, Lemma 2.3) — for any $\boldsymbol\beta, \mathbf{y}$, the proximal-gradient step $T(\boldsymbol\beta)$ satisfies $F(T(\boldsymbol\beta)) - F(\mathbf{y}) \le (L/2)[\|\boldsymbol\beta - \mathbf{y}\|^2 - \|T(\boldsymbol\beta) - \mathbf{y}\|^2]$ — combined with telescoping. Applied at iterate $k$ with $\mathbf{y} = \boldsymbol\beta^*$, the right side telescopes and the left side combines with monotonicity (descent lemma applied with $\mathbf{y} = \boldsymbol\beta^k$) to give the rate. $\blacksquare$

The $O(1/k)$ rate is sublinear: halving the suboptimality requires doubling $k$. ISTA is simple and stable but slow.


### §3.4 FISTA: Nesterov momentum and the $O(1/k^2)$ rate

Beck-Teboulle (2009) showed that adding **Nesterov momentum** to ISTA accelerates the rate from $O(1/k)$ to $O(1/k^2)$.

> **FISTA.** Set $\boldsymbol\beta^0 = \boldsymbol\beta^{-1} = \mathbf{0}$, $t_1 = 1$. For $k = 1, 2, \dots$:
> 1. $\mathbf{z}^k = \boldsymbol\beta^k + \frac{t_k - 1}{t_{k+1}} (\boldsymbol\beta^k - \boldsymbol\beta^{k-1})$ — momentum extrapolation.
> 2. $\boldsymbol\beta^{k+1} = S\!\left( \mathbf{z}^k - \frac{1}{nL} \mathbf{X}^\top (\mathbf{X} \mathbf{z}^k - \mathbf{y}), \; \lambda / L \right)$ — proximal gradient step from $\mathbf{z}^k$, not $\boldsymbol\beta^k$.
> 3. $t_{k+1} = (1 + \sqrt{1 + 4 t_k^2}) / 2$.

The momentum coefficient $(t_k - 1)/t_{k+1} \to 1$ as $k$ grows, giving the iteration a "running start" along the previous direction.

> **Theorem 3.3 (Beck-Teboulle 2009, Theorem 4.4).** With step size $1/L$,
> $$
> F(\boldsymbol\beta^k) - F(\boldsymbol\beta^*) \le \frac{2 L \|\boldsymbol\beta^0 - \boldsymbol\beta^*\|^2}{(k+1)^2} \quad \text{for all } k \ge 1.
> $$

**Proof.** Let $\mathbf{u}^k = t_k \boldsymbol\beta^k - (t_k - 1) \boldsymbol\beta^{k-1}$ and define the **Lyapunov function**

$$
v_k = \frac{2}{L} t_k^2 (F(\boldsymbol\beta^k) - F^*) + \|\mathbf{u}^k - \boldsymbol\beta^*\|^2.
$$

*Step 1 (Lyapunov non-increasing).* Apply the proximal-gradient inequality at $\mathbf{z}^k$ with $\mathbf{y} = \boldsymbol\beta^*$ and $\mathbf{y} = \boldsymbol\beta^k$. Multiply the first by $t_{k+1}$ and the second by $(t_{k+1} - 1)$ — invoking the FISTA recursion $t_{k+1}^2 - t_{k+1} = t_k^2$ — and add. After algebraic manipulation using the definition of $\mathbf{z}^k$, the right side telescopes to $\|\mathbf{u}^k - \boldsymbol\beta^*\|^2 - \|\mathbf{u}^{k+1} - \boldsymbol\beta^*\|^2$, and the left side becomes $(2/L)[t_{k+1}^2 (F(\boldsymbol\beta^{k+1}) - F^*) - t_k^2 (F(\boldsymbol\beta^k) - F^*)]$. Rearranging gives $v_{k+1} \le v_k$.

*Step 2 ($t_k \ge (k+1)/2$).* By induction. Base: $t_1 = 1 \ge 1$. Step: $t_{k+1} = (1 + \sqrt{1 + 4 t_k^2})/2 \ge (1 + 2 t_k)/2 \ge (k+2)/2$.

*Step 3.* From $v_k \le v_1 \le \|\boldsymbol\beta^0 - \boldsymbol\beta^*\|^2$ (the last bound is one ISTA descent step from the initialization) and $t_k \ge (k+1)/2$, we get $F(\boldsymbol\beta^k) - F^* \le L \|\boldsymbol\beta^0 - \boldsymbol\beta^*\|^2 / (2 t_k^2) \le 2 L \|\boldsymbol\beta^0 - \boldsymbol\beta^*\|^2 / (k+1)^2$. $\blacksquare$

A factor-of-$k$ improvement over ISTA. To reduce $F - F^*$ by 100×, ISTA needs 100× more iterations; FISTA needs only 10×.

**FISTA is not a descent method** — $F(\boldsymbol\beta^k)$ ripples slightly along iterations. A monotone variant (M-FISTA) accepts $\boldsymbol\beta^{k+1}$ only when $F$ decreases, but the trade-off is rarely worth it.


In [ ]:
# §§3.2-3.4 — Hand-rolled lasso solvers and convergence-rate comparison.

def lasso_loss(X, y, beta, lam):
    """The lasso objective F(beta) = (1/2n) ||y - X beta||^2 + lam ||beta||_1."""
    n = X.shape[0]
    return float(0.5 / n * np.sum((y - X @ beta) ** 2) + lam * np.sum(np.abs(beta)))


def ista(X, y, lam, max_iter=300, tol=0.0):
    """Iterative Soft-Thresholding Algorithm. Step size 1/L, L = ||X||_2^2/n."""
    n, p = X.shape
    L = float(np.linalg.norm(X, ord=2) ** 2 / n)
    eta = 1.0 / L
    beta = np.zeros(p)
    history = [lasso_loss(X, y, beta, lam)]
    for _ in range(max_iter):
        grad = X.T @ (X @ beta - y) / n
        beta_new = soft_threshold(beta - eta * grad, eta * lam)
        if tol > 0 and np.linalg.norm(beta_new - beta) < tol:
            beta = beta_new
            history.append(lasso_loss(X, y, beta, lam))
            break
        beta = beta_new
        history.append(lasso_loss(X, y, beta, lam))
    return beta, np.array(history)


def fista(X, y, lam, max_iter=300, tol=0.0):
    """FISTA (Beck-Teboulle 2009). Same setup as ISTA + Nesterov momentum."""
    n, p = X.shape
    L = float(np.linalg.norm(X, ord=2) ** 2 / n)
    eta = 1.0 / L
    beta_prev = np.zeros(p)
    beta = np.zeros(p)
    z = np.zeros(p)
    t = 1.0
    history = [lasso_loss(X, y, beta, lam)]
    for _ in range(max_iter):
        grad = X.T @ (X @ z - y) / n
        beta_new = soft_threshold(z - eta * grad, eta * lam)
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t * t)) / 2.0
        z = beta_new + ((t - 1.0) / t_new) * (beta_new - beta)
        if tol > 0 and np.linalg.norm(beta_new - beta) < tol:
            beta = beta_new
            history.append(lasso_loss(X, y, beta, lam))
            break
        beta_prev, beta, t = beta, beta_new, t_new
        history.append(lasso_loss(X, y, beta, lam))
    return beta, np.array(history)


def coord_descent(X, y, lam, max_iter=300, tol=0.0):
    """Cyclic coordinate descent for the lasso with incremental residual update."""
    n, p = X.shape
    col_norms = (X ** 2).sum(axis=0) / n      # c_j = ||X_j||^2/n
    beta = np.zeros(p)
    residual = y.copy()
    history = [lasso_loss(X, y, beta, lam)]
    for _ in range(max_iter):
        beta_old = beta.copy()
        for j in range(p):
            partial = residual + X[:, j] * beta[j]   # add j-th contribution back
            z_j = X[:, j] @ partial / n
            beta_new_j = soft_threshold(z_j, lam) / col_norms[j]
            residual -= X[:, j] * (beta_new_j - beta[j])
            beta[j] = beta_new_j
        history.append(lasso_loss(X, y, beta, lam))
        if tol > 0 and np.linalg.norm(beta - beta_old) < tol:
            break
    return beta, np.array(history)


# --- Run all three on the §1 DGP at lambda = lambda_CV ---
n_iter = 300
lam = lam_cv

# High-precision reference for F* — FISTA with many iterations.
print("Computing reference F* with FISTA at 5000 iterations...")
beta_ref, _ = fista(X1, y1, lam, max_iter=5000, tol=0)
F_star = lasso_loss(X1, y1, beta_ref, lam)
print(f"Reference F* = {F_star:.10f}")

print(f"\nRunning ISTA, FISTA, CD for {n_iter} iterations each...")
_, hist_ista  = ista (X1, y1, lam, max_iter=n_iter, tol=0)
_, hist_fista = fista(X1, y1, lam, max_iter=n_iter, tol=0)
_, hist_cd    = coord_descent(X1, y1, lam, max_iter=n_iter, tol=0)

# Suboptimality F(beta_k) - F*.
gap_ista  = np.maximum(hist_ista  - F_star, 1e-18)
gap_fista = np.maximum(hist_fista - F_star, 1e-18)
gap_cd    = np.maximum(hist_cd    - F_star, 1e-18)

# Sanity assertions: FISTA should beat ISTA at every k > ~10 by a wide margin;
# CD should be competitive with FISTA after the active set stabilizes.
k_check = 100
print(f"\nAt iteration {k_check}:")
print(f"  ISTA  gap = {gap_ista[k_check]:.3e}")
print(f"  FISTA gap = {gap_fista[k_check]:.3e}")
print(f"  CD    gap = {gap_cd[k_check]:.3e}")
assert gap_fista[k_check] < 0.1 * gap_ista[k_check], \
    "FISTA should beat ISTA by >=10x at iteration 100"
assert gap_cd[k_check] < 0.5 * gap_ista[k_check], \
    "CD should also beat ISTA by a wide margin at iteration 100"

# --- Figure 3.1: convergence trace, log-log axes ---
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ks = np.arange(len(gap_ista))
ax.loglog(ks[1:], gap_ista[1:],  color=COLOR_OLS,    linewidth=1.8,
          label=fr"ISTA  ($O(1/k)$, observed slope $\approx -1$)")
ax.loglog(ks[1:], gap_fista[1:], color=COLOR_LASSO,  linewidth=1.8,
          label=fr"FISTA ($O(1/k^2)$, observed slope $\approx -2$)")
ax.loglog(ks[1:], gap_cd[1:],    color=COLOR_RIDGE,  linewidth=1.8,
          label="Coordinate descent (linear after active set stabilizes)")

# Reference slopes -1 and -2 for visual comparison.
k_ref = ks[10:]
ax.loglog(k_ref, gap_ista[10] * (10.0 / k_ref) ** 1.0, "--",
          color=COLOR_OLS, linewidth=0.8, alpha=0.5)
ax.loglog(k_ref, gap_fista[10] * (10.0 / k_ref) ** 2.0, "--",
          color=COLOR_LASSO, linewidth=0.8, alpha=0.5)

ax.set_xlabel("Iteration $k$")
ax.set_ylabel(r"$F(\boldsymbol{\beta}^k) - F^*$  (suboptimality)")
ax.set_title(f"Convergence on §1 DGP at $\\lambda = \\lambda_{{\\mathrm{{CV}}}}$ "
             f"({lam:.4f})  --  $n = {X1.shape[0]}$, $p = {X1.shape[1]}$")
ax.legend(loc="lower left", fontsize=9)
ax.grid(True, which="both", linewidth=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_03_01_solver_convergence.png", bbox_inches="tight")
plt.show()


### §3.5 Practical solver-choice notes

A field guide to which solver to reach for.

**Coordinate descent (`sklearn.linear_model.Lasso`, `glmnet`).** Default for everything in the lasso family — lasso, elastic net, group lasso. Fastest in practice for $n, p \le 10^5$ at moderate sparsity. Warm starts along a $\lambda$ path are nearly free, which is why `LassoCV` is fast even with 100-fold path × 10-fold CV.

**FISTA.** The right default for lasso variants where the L1 prox is easy but coordinate-by-coordinate updates are not — group lasso with overlapping groups, fused lasso, generalized lasso with non-axis-aligned penalty, total-variation penalties. Also the right choice when $\mathbf{X}$ is structured (FFT, wavelet) and $\mathbf{X} \mathbf{v}$ admits a fast algorithm — coordinate descent breaks the structured-multiplication advantage by accessing one column at a time.

**ISTA.** Pedagogically valuable, rarely the right algorithmic choice — FISTA dominates it at no extra implementation cost. Use ISTA only when the proof of correctness or the descent property is needed (some monotonicity-sensitive applications in statistical guarantees that rely on objective decrease).

**Specialized solvers we don't cover.** ADMM (Boyd et al. 2011) for lasso variants with linear-equality-coupled penalties (e.g., the Dantzig selector). LARS (Efron-Hastie-Johnstone-Tibshirani 2004) computes the entire lasso path exactly in $O(np \min(n, p))$, which can beat coordinate descent at very small $p$ but loses badly at high-dimensional scales. Interior-point methods (`cvxpy`, `cvxopt`) work but are typically 100× slower than CD on lasso problems of any meaningful size.

For everything in the rest of this topic — the §1 lasso fits, the LassoCV in §7, the elastic-net comparison in §8, the debiased-lasso pipeline in §10 — we use scikit-learn's coordinate descent. We hand-rolled FISTA above to demonstrate the $O(1/k^2)$ rate and to keep the algorithmic content visible.


## §4. Bias-variance for the lasso

The lasso's central trade-off is between **bias** (from L1 shrinkage of active coefficients) and **variance** (controlled by the size of the data-adapted active set). As $\lambda$ ranges from $0$ to $\lambda_{\max}$, the prediction risk traces the canonical U-curve. This section formalizes both halves, computes $\lambda_{\max}$ in closed form from the KKT conditions, and develops the lasso solution path $\hat{\boldsymbol\beta}(\lambda)$ as a piecewise-linear function of $\lambda$.

The U-curve is the practical bridge between §3 (we can compute the lasso) and §5 (the oracle inequality bounds the bottom of the U). The solution path is what `LassoCV` operates on when it picks a $\lambda$.


### §4.1 The bias contribution from L1 shrinkage

The lasso's shrinkage isn't soft and asymptotically vanishing — it's a **constant absolute shrinkage** that biases every active coefficient toward zero by approximately $\lambda$.

**The orthogonal case makes this explicit.** From §3.1 with $\mathbf{X}^\top \mathbf{X} = n \mathbf{I}$ and $z_j = (\mathbf{X}^\top \mathbf{y} / n)_j$, $\hat\beta_j = S(z_j, \lambda)$. Under the model $z_j \sim \mathcal{N}(\beta^*_j, \sigma^2/n)$:

- **Large-signal coordinates** ($|\beta^*_j| \gg \lambda + \sigma/\sqrt{n}$): $\mathbb{E}[\hat\beta_j] \approx \beta^*_j - \lambda \, \mathrm{sign}(\beta^*_j)$. The bias is $-\lambda \, \mathrm{sign}(\beta^*_j)$ — constant magnitude, opposite sign to the true value, independent of how large $\beta^*_j$ is.
- **Noise coordinates** ($\beta^*_j = 0$): $\mathbb{E}[\hat\beta_j] = 0$ by symmetry of $z_j \sim \mathcal{N}(0, \sigma^2/n)$. No bias, but a small variance contribution from false positives.

For general (non-orthogonal) designs, conditioning on correct active-set identification:

$$
\hat{\boldsymbol\beta}_S = \hat{\boldsymbol\beta}_S^{\text{OLS-on-}S} - \lambda \big(\mathbf{X}_S^\top \mathbf{X}_S / n\big)^{-1} \mathrm{sign}(\hat{\boldsymbol\beta}_S),
$$

where $\hat{\boldsymbol\beta}_S^{\text{OLS-on-}S}$ is the OLS estimator on the active features. The shrinkage correction scales linearly in $\lambda$.

This bias is the price of sparsity. Two later sections fix it for different reasons: the **adaptive lasso** (Zou 2006, §8.3) uses feature-specific weights so that the active-coefficient bias decays asymptotically; the **debiased lasso** (§10.2) explicitly subtracts off the shrinkage bias via a one-step correction. For prediction, the bias isn't catastrophic — the U-curve in §4.3 shows the trade-off is favorable at well-chosen $\lambda$. For inference, the bias is the central problem of the topic.


### §4.2 Variance from sparsity adaptation

In contrast to the bias, the lasso's **variance** is small — much smaller than OLS variance at the same $p$, when OLS is even defined.

OLS variance scales with the number of features: $\mathbb{V}\mathrm{ar}(\hat{\boldsymbol\beta}^{\text{OLS}}) = \sigma^2 (\mathbf{X}^\top \mathbf{X})^{-1}$, with trace scaling as $\sigma^2 p / n$ for a well-conditioned design. As $p \to n$, the variance blows up.

The lasso, by zeroing out coordinates with small $|z_j|$, effectively reduces the model dimension to $|\hat A_\lambda|$. The **degrees of freedom of the lasso** (Zou-Hastie-Tibshirani 2007) is exactly $\mathrm{df}(\hat{\boldsymbol\beta}^{\text{lasso}}(\lambda)) = \mathbb{E}[|\hat A_\lambda|]$. So the lasso's prediction variance scales as $\sigma^2 \mathbb{E}[|\hat A_\lambda|] / n$, which for well-chosen $\lambda$ is on the order of $\sigma^2 s / n$ — proportional to the **true** sparsity, not to $p$.

This is the **sparsity-adaptation property**: the lasso pays a variance cost proportional to the model size it actually uses, regardless of how many candidate features were available. It's the central reason the lasso works in the $p \gg n$ regime where OLS doesn't.

For DGP-1 with $s = 10$, $\sigma = 0.5$, $n = 200$: lasso variance $\sim \sigma^2 s / n = 0.0125$, OLS variance at $p = 199$ would be $\sim \sigma^2 \cdot 199 / 200 \approx 0.249$ — a 20× advantage, before counting bias.


### §4.3 The U-curve as $\lambda$ varies

The bias and variance pieces combine into the canonical U-shaped prediction-risk curve. Define the **integrated prediction risk** $\mathrm{IPE}(\lambda) = \mathbb{E}_{\mathbf{x}^*}[(\mathbf{x}^{*\top} \hat{\boldsymbol\beta}(\lambda) - \mathbf{x}^{*\top} \boldsymbol\beta^*)^2] = \mathrm{Bias}^2(\lambda) + \mathrm{Variance}(\lambda)$, with expectation over training-set draws (test point $\mathbf{x}^*$ fixed).

Two boundaries:
- **$\lambda = 0$:** bias near-zero, variance dominates (or undefined for OLS at $p > n$).
- **$\lambda = \lambda_{\max}$:** variance zero ($\hat{\boldsymbol\beta} = \mathbf{0}$ deterministically), bias² equals the full prediction signal.

Between them, the curve is U-shaped with an optimal $\lambda^*$ that minimizes IPE. The §5 oracle inequality bounds this minimum from above by $C \sigma^2 s \log(p) / n$.

**$\lambda_{\max}$ in closed form.** From the KKT conditions of §2.4, $\hat{\boldsymbol\beta} = \mathbf{0}$ is a lasso solution if and only if $|(\mathbf{X}^\top \mathbf{y} / n)_j| \le \lambda$ for all $j$, so

$$
\lambda_{\max} = \left\| \frac{\mathbf{X}^\top \mathbf{y}}{n} \right\|_\infty = \max_j \left| \frac{\mathbf{X}_j^\top \mathbf{y}}{n} \right|.
$$

For $\lambda \ge \lambda_{\max}$, the lasso solution is identically zero. For $\lambda$ just below $\lambda_{\max}$, exactly one coordinate becomes active. We compute $\lambda_{\max}$ on DGP-1 below (~1.04, two orders of magnitude above $\lambda_{\text{CV}} \approx 0.06$).


In [ ]:
# §4.3 — Empirical U-curve on DGP-1 via Monte Carlo.
# B replicate training sets, fixed test set, lasso path at each replicate,
# decompose prediction MSE on test into bias^2 + variance.

from sklearn.linear_model import lasso_path

# lambda_max for DGP-1 (using the §1 sample as a representative case).
lambda_max = float(np.max(np.abs(X1.T @ y1)) / X1.shape[0])
print(f"lambda_max for §1 sample: {lambda_max:.4f}")
print(f"lambda_CV for §1 sample:  {lam_cv:.4f}  (ratio lambda_CV/lambda_max = {lam_cv/lambda_max:.4f})")

# Setup: B = 50 MC replicates, decreasing lambda-grid for warm-start efficiency.
B_mc = 50
n_test = 500
lambda_grid = np.logspace(np.log10(lambda_max * 1.01), -2.5, 30)  # decreasing

# Fixed test set (held constant across MC replicates).
X_test, y_test, beta_star_mc, _ = make_dgp1(
    n=n_test, rng=np.random.default_rng(SEED + 300))
true_preds = X_test @ beta_star_mc

# Storage: predictions[b, k, i] = lasso prediction at test point i, replicate b, lambda-index k.
predictions = np.zeros((B_mc, len(lambda_grid), n_test))
for b in range(B_mc):
    X_train_b, y_train_b, _, _ = make_dgp1(rng=np.random.default_rng(SEED + 1000 + b))
    _, coefs, _ = lasso_path(X_train_b, y_train_b, alphas=lambda_grid,
                              tol=1e-7, max_iter=20_000)
    # coefs shape: (p, n_alphas). predictions[b] should be (n_alphas, n_test).
    predictions[b] = (X_test @ coefs).T

# Bias^2, variance, MSE.
mean_pred = predictions.mean(axis=0)                 # (n_lambda, n_test)
bias_sq   = np.mean((mean_pred - true_preds) ** 2, axis=1)
variance  = np.mean(predictions.var(axis=0), axis=1)
mse_total = bias_sq + variance

# Sanity: bias^2 monotonically decreases in lambda (= decreases along the
# array, since lambda_grid is decreasing). The MSE total should be U-shaped
# in lambda, with argmin in the interior. Variance is NOT monotonic in the
# n < p sparse regime — it is U-shaped on its own (small at very large lambda
# where beta_hat ~ 0; small in the lasso "sweet spot" where soft-thresholding
# stabilizes the support; large at very small lambda where the lasso
# approaches the non-unique OLS solution and variance explodes), so we do not
# assert variance monotonicity here. The variance trajectory itself is part
# of what Figure 4.1 below visualizes.
assert np.sum(np.diff(bias_sq) > 1e-3) <= 4, \
    "bias^2 should be (approximately) monotonically increasing in lambda"
mse_argmin = int(np.argmin(mse_total))
assert 0 < mse_argmin < len(mse_total) - 1, \
    "MSE should be U-shaped — argmin should sit in the interior of the lambda grid"

# Find empirical lambda-min.
lam_min_emp = lambda_grid[int(np.argmin(mse_total))]
print(f"\nEmpirical MSE-minimizing lambda from MC: {lam_min_emp:.4f}")
print(f"LassoCV-selected lambda from §1:         {lam_cv:.4f}")

# --- Figure 4.1 — U-curve ---
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.plot(lambda_grid, bias_sq, "o-", color=COLOR_RIDGE,
        linewidth=1.6, markersize=4, label=r"Bias$^2$")
ax.plot(lambda_grid, variance, "s-", color=COLOR_LASSO,
        linewidth=1.6, markersize=4, label="Variance")
ax.plot(lambda_grid, mse_total, "-", color=COLOR_TRUTH,
        linewidth=2.2, label=r"MSE = Bias$^2$ + Variance")
ax.axvline(lam_cv, color=COLOR_DEBIASED, linewidth=1.3, linestyle="--",
           label=fr"$\lambda_{{\mathrm{{CV}}}} = {lam_cv:.4f}$")
ax.axvline(lambda_max, color="#888888", linewidth=0.9, linestyle=":",
           label=fr"$\lambda_{{\max}} = {lambda_max:.3f}$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\lambda$ (log scale)")
ax.set_ylabel(r"prediction MSE on test set (log scale)")
ax.set_title(f"Lasso U-curve on §1 DGP --- $B = {B_mc}$ MC replicates, $n_{{\\mathrm{{test}}}} = {n_test}$")
ax.legend(loc="lower left", fontsize=9)
ax.grid(True, which="both", linewidth=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_04_01_u_curve.png", bbox_inches="tight")
plt.show()


### §4.4 The lasso solution path is piecewise linear

The **lasso solution path** is the function $\lambda \mapsto \hat{\boldsymbol\beta}(\lambda)$ for $\lambda \in [0, \lambda_{\max}]$.

> **Theorem 4.1 (piecewise linearity; Efron-Hastie-Johnstone-Tibshirani 2004).** The lasso solution path $\hat{\boldsymbol\beta}(\lambda)$ is a continuous piecewise-linear function of $\lambda$. There is a finite sequence of **knots** $\lambda_{\max} = \lambda_{(0)} > \lambda_{(1)} > \dots > \lambda_{(K)} = 0$ such that on each interval $[\lambda_{(k+1)}, \lambda_{(k)}]$, the active set $\hat A_\lambda$ is constant and $\hat{\boldsymbol\beta}(\lambda)$ is linear in $\lambda$.

The proof is direct from KKT: on each interval where $\hat A_\lambda$ is fixed, the active condition $\frac{1}{n}\mathbf{X}_{\hat A}^\top(\mathbf{y} - \mathbf{X}\hat{\boldsymbol\beta}) = \lambda \, \mathrm{sign}(\hat{\boldsymbol\beta}_{\hat A})$ is a linear system in $\hat{\boldsymbol\beta}_{\hat A}$ with $\lambda$ on the right-hand side, so the solution is linear in $\lambda$. The LARS algorithm (Efron et al. 2004) traces this knot-by-knot in $O(np \min(n, p))$ total time; for moderate-to-large $p$ coordinate descent on a $\lambda$-grid is faster in practice.

**Reading the path.** Plotting $\hat\beta_j(\lambda)$ vs $\log \lambda$ for all $j$: as $\lambda$ decreases from $\lambda_{\max}$, the first coordinate to enter is $\arg\max_j |(\mathbf{X}^\top \mathbf{y} / n)_j|$ — the feature most correlated with the response. Subsequent coordinates enter at successively smaller $\lambda$, roughly in order of importance. A coordinate can also leave the active set (coefficient passes through zero), uncommon in continuous designs but possible.


In [ ]:
# §4.4 — Lasso solution path on the §1 DGP.

# Compute the path on a fine lambda-grid, using sklearn's lasso_path with warm starts.
lambda_path = np.logspace(np.log10(lambda_max * 1.01), np.log10(lambda_max * 1e-3), 100)
_, path_coefs, _ = lasso_path(X1, y1, alphas=lambda_path,
                               tol=1e-8, max_iter=20_000)
# path_coefs shape: (p, n_lambda)

# Sanity: at lambda = lambda_max (largest in our grid), the solution should be all zeros
# (or have at most 1 active coefficient).
n_active_at_max = int(np.sum(np.abs(path_coefs[:, 0]) > 1e-10))
print(f"Active coordinates at lambda ~ lambda_max: {n_active_at_max}  (expect 0 or 1)")
assert n_active_at_max <= 1, "At lambda_max the lasso solution should be identically zero or have one active coord"

# Order in which coordinates enter the active set (smallest lambda at which |beta_hat_j| > 1e-10).
entry_lambda = np.full(X1.shape[1], np.nan)
for j in range(X1.shape[1]):
    nonzero = np.where(np.abs(path_coefs[j, :]) > 1e-10)[0]
    if len(nonzero) > 0:
        entry_lambda[j] = lambda_path[nonzero[0]]
order = np.argsort(-entry_lambda)  # decreasing lambda = first to enter
print("First 10 coords to enter the active set (and their entry lambda):")
for j in order[:10]:
    in_S = "* true active" if j < 10 else "  noise"
    print(f"  j = {j:>3d}  enters at lambda = {entry_lambda[j]:.4f}  {in_S}")

# --- Figure 4.2 — Solution path ---
fig, ax = plt.subplots(figsize=(9.0, 5.5))
# Inactive coords (j >= 10): light gray, thin lines.
for j in range(10, X1.shape[1]):
    ax.plot(lambda_path, path_coefs[j, :], color=COLOR_OLS,
            alpha=0.30, linewidth=0.5)
# Active coords (j in S = {0..9}): black, thick lines, labeled.
for j in range(10):
    ax.plot(lambda_path, path_coefs[j, :], color=COLOR_TRUTH,
            alpha=0.95, linewidth=1.8)

ax.axvline(lam_cv, color=COLOR_DEBIASED, linewidth=1.4, linestyle="--",
           label=fr"$\lambda_{{\mathrm{{CV}}}} = {lam_cv:.4f}$")
ax.axvline(lambda_max, color="#666666", linewidth=0.9, linestyle=":",
           label=fr"$\lambda_{{\max}} = {lambda_max:.3f}$")
ax.axhline(1.0, color="#aaaaaa", linewidth=0.7, linestyle=":",
           alpha=0.6, label=r"$\beta^*_j = 1$ (true active value)")
ax.axhline(0, color="black", linewidth=0.4)

ax.set_xscale("log")
ax.set_xlabel(r"$\lambda$ (log scale)")
ax.set_ylabel(r"$\hat\beta_j(\lambda)$")
ax.set_title("Lasso solution path on §1 DGP --- true active $j \\in S$ in black, "
             "noise coords $j \\notin S$ in gray")
ax.legend(loc="upper left", fontsize=9)
ax.invert_xaxis()  # convention: lambda decreasing left-to-right (more features as we move right)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_04_02_solution_path.png", bbox_inches="tight")
plt.show()


## §5. The lasso oracle inequality

This is the topic's headline theoretical result: under a restricted-eigenvalue condition on the design and a deviation bound on the noise, the lasso achieves prediction risk of order $\sigma^2 s \log(p) / n$ — comparable to what an oracle that knew the true active set $S$ in advance could achieve, up to a logarithmic factor in $p$. The bound is **non-asymptotic** (holds with high probability for any finite $n$, $p$), **dimension-free** (the dependence on $p$ is only logarithmic), and **rate-optimal** for sparse high-dimensional regression.

The proof has four steps and follows Bickel-Ritov-Tsybakov (2009). **Step 1 (the basic inequality)** uses the lasso's optimality to bound prediction error in terms of the L1 estimation error and a noise term. **Step 2 (the cone condition)** shows the error vector $\hat{\boldsymbol\beta} - \boldsymbol\beta^*$ has most of its L1 mass on $S$. **Step 3 (the restricted-eigenvalue condition)** lets us convert L1 estimation error on $S$ into a lower bound on the prediction error. **Step 4 (the deviation step)** controls the noise term using a maximal sub-Gaussian inequality.


### §5.1 Setup: prediction risk in the high-dim regime

Standard high-dimensional linear regression model: $\mathbf{y} = \mathbf{X} \boldsymbol\beta^* + \boldsymbol\varepsilon$, $\mathbf{X} \in \mathbb{R}^{n \times p}$, $\boldsymbol\beta^* \in \mathbb{R}^p$, $\boldsymbol\varepsilon \in \mathbb{R}^n$. Assumptions:

- **Sparsity:** $\boldsymbol\beta^*$ has support $S$ with $|S| = s$, $s \ll p$.
- **Sub-Gaussian noise:** independent $\varepsilon_i$ with sub-Gaussian parameter $\sigma$ ($\mathbb{E}[\exp(t\varepsilon_i)] \le \exp(t^2 \sigma^2/2)$). Gaussian noise is the canonical case.
- **Column-normalized design:** $\|\mathbf{X}_j\|_2^2 \le n$ for all $j$ (equivalently $(1/n)\|\mathbf{X}_j\|_2^2 \le 1$).

The **prediction risk** is $\mathrm{PE}(\hat{\boldsymbol\beta}) = (1/n) \|\mathbf{X}(\hat{\boldsymbol\beta} - \boldsymbol\beta^*)\|_2^2$ — the average squared in-sample prediction error against the true regression function.

The **oracle estimator** is OLS restricted to the (known) true support: $\hat{\boldsymbol\beta}^{\text{oracle}}_S = (\mathbf{X}_S^\top \mathbf{X}_S)^{-1} \mathbf{X}_S^\top \mathbf{y}$, $\hat{\boldsymbol\beta}^{\text{oracle}}_{S^c} = \mathbf{0}$. Its risk is $\sigma^2 s / n$ in expectation. The oracle inequality says the lasso achieves the same rate up to a $\log(p)$ factor — without knowing $S$.


### §5.2 The basic inequality

Starting point: lasso optimality, $\hat{\boldsymbol\beta}$ does no worse than $\boldsymbol\beta^*$:

$$
\tfrac{1}{2n}\|\mathbf{y} - \mathbf{X}\hat{\boldsymbol\beta}\|^2 + \lambda\|\hat{\boldsymbol\beta}\|_1 \le \tfrac{1}{2n}\|\mathbf{y} - \mathbf{X}\boldsymbol\beta^*\|^2 + \lambda\|\boldsymbol\beta^*\|_1.
$$

Substitute $\mathbf{y} = \mathbf{X}\boldsymbol\beta^* + \boldsymbol\varepsilon$, let $\boldsymbol\delta = \hat{\boldsymbol\beta} - \boldsymbol\beta^*$, expand and cancel $(1/2n)\|\boldsymbol\varepsilon\|^2$:

$$
\tfrac{1}{n}\|\mathbf{X}\boldsymbol\delta\|^2 \le \tfrac{2}{n}\boldsymbol\varepsilon^\top \mathbf{X}\boldsymbol\delta + 2\lambda(\|\boldsymbol\beta^*\|_1 - \|\hat{\boldsymbol\beta}\|_1). \tag{$\star$}
$$

**Control the noise term via Hölder.** $(2/n)|\boldsymbol\varepsilon^\top \mathbf{X}\boldsymbol\delta| \le 2 \|\mathbf{X}^\top\boldsymbol\varepsilon/n\|_\infty \cdot \|\boldsymbol\delta\|_1$. Define the **noise event** $\mathcal{E} = \{\|\mathbf{X}^\top\boldsymbol\varepsilon/n\|_\infty \le \lambda/2\}$. On $\mathcal{E}$, $(2/n)\boldsymbol\varepsilon^\top \mathbf{X}\boldsymbol\delta \le \lambda \|\boldsymbol\delta\|_1$.

**Control the L1-norm difference via the support split.** Decompose $\hat{\boldsymbol\beta} = \hat{\boldsymbol\beta}_S + \hat{\boldsymbol\beta}_{S^c}$. Since $\boldsymbol\beta^*_{S^c} = \mathbf{0}$:

$$
\|\boldsymbol\beta^*\|_1 - \|\hat{\boldsymbol\beta}\|_1 = \|\boldsymbol\beta^*_S\|_1 - \|\hat{\boldsymbol\beta}_S\|_1 - \|\hat{\boldsymbol\beta}_{S^c}\|_1 \le \|\boldsymbol\delta_S\|_1 - \|\boldsymbol\delta_{S^c}\|_1
$$

(reverse triangle on $\hat{\boldsymbol\beta}_S$ vs $\boldsymbol\beta^*_S$). Substitute everything into $(\star)$:

$$
\boxed{\;\tfrac{1}{n}\|\mathbf{X}\boldsymbol\delta\|^2 + \lambda \|\boldsymbol\delta_{S^c}\|_1 \le 3\lambda \|\boldsymbol\delta_S\|_1\;} \quad \text{on } \mathcal{E}. \tag{$\star\star$}
$$


### §5.3 The cone condition

Drop the non-negative prediction-error term from $(\star\star)$:

$$
\boxed{\|\boldsymbol\delta_{S^c}\|_1 \le 3 \|\boldsymbol\delta_S\|_1.}
$$

The error vector lies in the convex cone $\mathcal{C}(S, 3) := \{\boldsymbol\delta : \|\boldsymbol\delta_{S^c}\|_1 \le 3 \|\boldsymbol\delta_S\|_1\}$ — most of its L1 mass is on the true support.

This is the structural content of the basic inequality: without any further assumption on $\mathbf{X}$ or $\boldsymbol\varepsilon$ beyond $\mathcal{E}$, the lasso error is concentrated on $S$. Converting this into a rate bound requires a design assumption — the next step.


### §5.4 The restricted-eigenvalue condition

The basic inequality $(\star\star)$ gives $(1/n)\|\mathbf{X}\boldsymbol\delta\|^2 \le 3\lambda \|\boldsymbol\delta_S\|_1$. By Cauchy-Schwarz on $\boldsymbol\delta_S \in \mathbb{R}^s$, $\|\boldsymbol\delta_S\|_1 \le \sqrt{s} \|\boldsymbol\delta_S\|_2$. To convert this into a prediction-risk rate, we need a *lower bound* on $(1/n)\|\mathbf{X}\boldsymbol\delta\|^2$ in terms of $\|\boldsymbol\delta_S\|^2$.

> **Definition 5.1 (restricted-eigenvalue; Bickel-Ritov-Tsybakov 2009).** $\mathbf{X}$ satisfies $\mathrm{RE}(s, c_0)$ with constant $\kappa > 0$ if for every $\boldsymbol\delta \in \mathcal{C}(S, c_0)$ with $|S| \le s$,
> $$
> \tfrac{1}{n}\|\mathbf{X}\boldsymbol\delta\|^2 \ge \kappa^2 \|\boldsymbol\delta_S\|_2^2.
> $$

In words: on the cone, the empirical Gram matrix $\mathbf{X}^\top\mathbf{X}/n$ acts like a positive-definite matrix on the active block, with smallest "restricted eigenvalue" at least $\kappa^2$. On the full $\mathbb{R}^p$ this would be $\lambda_{\min}(\mathbf{X}^\top\mathbf{X}/n) = 0$ when $p > n$; the cone restriction is what makes RE feasible in the high-dim regime.

**When does RE hold?**
- **Random Gaussian designs** with $\lambda_{\min}(\boldsymbol\Sigma) \ge \kappa_0^2$ and $n \gtrsim s\log(p)$: RE holds with $\kappa^2 \asymp \kappa_0^2$ w.h.p. (Raskutti-Wainwright-Yu 2010).
- **Sub-Gaussian designs:** same conclusion (Rudelson-Zhou 2013).
- **RIP $\Rightarrow$ RE** (Candès-Tao 2005; covered in §9).

For DGP-1 with AR(1) Toeplitz $\boldsymbol\Sigma$ and $\rho = 0.5$, $\lambda_{\min}(\boldsymbol\Sigma) \to (1-\rho)/(1+\rho) = 1/3$ as $p \to \infty$, so RE holds with $\kappa^2 \approx 1/3$.


### §5.5 The deviation step and the $O(\sigma^2 s \log p / n)$ rate

**Combining basic inequality and RE.** From $(\star\star)$ + Cauchy-Schwarz + RE:

$$
\tfrac{1}{n}\|\mathbf{X}\boldsymbol\delta\|^2 \le 3\lambda \sqrt{s}\|\boldsymbol\delta_S\|_2 \le \tfrac{3\lambda\sqrt{s}}{\kappa} \sqrt{\tfrac{1}{n}\|\mathbf{X}\boldsymbol\delta\|^2}.
$$

Let $u = \sqrt{(1/n)\|\mathbf{X}\boldsymbol\delta\|^2}$. Then $u^2 \le 3\lambda\sqrt{s} u/\kappa$, so $u \le 3\lambda\sqrt{s}/\kappa$. Squaring:

$$
\tfrac{1}{n}\|\mathbf{X}(\hat{\boldsymbol\beta} - \boldsymbol\beta^*)\|^2 \le \tfrac{9\lambda^2 s}{\kappa^2} \quad \text{on } \mathcal{E} \cap \{\mathrm{RE}(s,3)\}. \tag{$\star\!\star\!\star$}
$$

**The deviation step (sub-Gaussian maximal inequality).** For a single coordinate $j$, $(\mathbf{X}_j^\top \boldsymbol\varepsilon)/n$ is sub-Gaussian with parameter $\sigma_j^2 = \sigma^2 \|\mathbf{X}_j\|^2 / n^2 \le \sigma^2/n$. Sub-Gaussian tail + union bound over $p$ coords and 2 tails:

$$
\mathbb{P}\!\left(\|\mathbf{X}^\top\boldsymbol\varepsilon/n\|_\infty > t\right) \le 2p \exp(-nt^2/(2\sigma^2)).
$$

Setting RHS = $\delta$: $t = \sigma\sqrt{2\log(2p/\delta)/n}$. Choose $\lambda = 2\sigma\sqrt{2\log(2p/\delta)/n}$ — then $\lambda/2 \ge \|\mathbf{X}^\top\boldsymbol\varepsilon/n\|_\infty$ on $\mathcal{E}$, which holds with probability $\ge 1 - \delta$.

> **Theorem 5.1 (lasso oracle inequality; Bickel-Ritov-Tsybakov 2009).** Assume $\boldsymbol\beta^*$ is $s$-sparse, the noise has independent sub-Gaussian entries with parameter $\sigma$, the columns satisfy $\|\mathbf{X}_j\|^2 \le n$, and the design satisfies $\mathrm{RE}(s, 3)$ with constant $\kappa > 0$. Choose $\lambda = 2\sigma\sqrt{2\log(2p/\delta)/n}$. Then with probability at least $1 - \delta$,
> $$
> \tfrac{1}{n}\|\mathbf{X}(\hat{\boldsymbol\beta}^{\text{lasso}} - \boldsymbol\beta^*)\|^2 \le \tfrac{72 \sigma^2 s \log(2p/\delta)}{n \kappa^2}.
> $$

Setting $\delta = 1/p$, $\log(2p/\delta) = \log(2p^2) \le 2\log(2p)$, so the rate is $\sigma^2 s \log(p)/(n\kappa^2)$ — the **fundamental rate** for sparse high-dim regression. The $\log(p)$ factor is the price of not knowing $S$; the rate is minimax-optimal up to this $\log(p)$ factor.


In [ ]:
# §5.5 — Numerical demonstration of the oracle-inequality rate.
# Hold p, s, sigma fixed; vary n; fit lasso at the theory-guided
# lambda = 2 * sigma * sqrt(2 log(p)/n); compute prediction risk on a fresh test set.
# Compare to the theoretical bound C * sigma^2 * s * log(p) / (n * kappa^2).

# Setup.
p_fix    = 500
s_fix    = 10
sigma_fx = 0.5
rho_fix  = 0.5
n_grid   = [50, 100, 200, 400, 800]
B_oi     = 30        # MC replicates per n
n_test_oi = 1000     # large test set for stable risk estimates

# RE constant lower bound: lambda_min(Sigma) -> (1 - rho)/(1 + rho) for AR(1) Sigma.
kappa_sq = (1 - rho_fix) / (1 + rho_fix)   # ~ 1/3 for rho=0.5
print(f"AR(1) RE constant lower bound: kappa^2 ~ {kappa_sq:.4f}")

emp_risk = np.zeros((len(n_grid), B_oi))
for i, n_train in enumerate(n_grid):
    # Theory-guided lambda for this n. delta = 1/p.
    lam_theory = 2 * sigma_fx * np.sqrt(2 * np.log(2 * p_fix / (1.0/p_fix)) / n_train)
    for b in range(B_oi):
        rng_b = np.random.default_rng(SEED + 5000 + 100*i + b)
        X_tr_oi, y_tr_oi, beta_star_oi, _ = make_dgp1(
            n=n_train, p=p_fix, s=s_fix, sigma=sigma_fx, rho=rho_fix, rng=rng_b)
        rng_te = np.random.default_rng(SEED + 6000 + 100*i + b)
        X_te_oi, _, _, _ = make_dgp1(
            n=n_test_oi, p=p_fix, s=s_fix, sigma=sigma_fx, rho=rho_fix, rng=rng_te)
        beta_hat = Lasso(alpha=lam_theory, fit_intercept=False, tol=1e-7,
                          max_iter=20_000).fit(X_tr_oi, y_tr_oi).coef_
        # Prediction risk: (1/n_test) ||X_test (beta_hat - beta*)||^2
        emp_risk[i, b] = float(np.mean((X_te_oi @ (beta_hat - beta_star_oi)) ** 2))

emp_risk_mean = emp_risk.mean(axis=1)
emp_risk_se   = emp_risk.std(axis=1, ddof=1) / np.sqrt(B_oi)

# Theoretical bound: C * sigma^2 * s * log(p) / (n * kappa^2) with C = 72 (BRT constant).
# We also plot a "calibrated" version with a smaller constant to show the slope match.
C_brt = 72.0
theory_brt = C_brt * sigma_fx**2 * s_fix * np.log(p_fix) / (np.array(n_grid) * kappa_sq)

# Calibrated constant: pick C so the bound passes through the empirical curve at the
# largest n. This shows the slope is correct even if BRT's C=72 is loose.
C_emp = emp_risk_mean[-1] * n_grid[-1] / (sigma_fx**2 * s_fix * np.log(p_fix))
theory_calibrated = C_emp * sigma_fx**2 * s_fix * np.log(p_fix) / np.array(n_grid)

print(f"\n{'n':>6}  {'emp risk':>12}  {'BRT bound':>12}  {'calib bound':>12}")
for i, n in enumerate(n_grid):
    print(f"{n:>6d}  {emp_risk_mean[i]:>12.4f}  {theory_brt[i]:>12.4f}  {theory_calibrated[i]:>12.4f}")

# Sanity: empirical risk should decrease as n grows, with slope close to -1 on log-log.
log_n = np.log(np.array(n_grid))
log_risk = np.log(emp_risk_mean)
slope, intercept = np.polyfit(log_n, log_risk, 1)
print(f"\nFitted slope of log(empirical risk) vs log(n): {slope:.3f}  (theory predicts -1)")
assert -1.5 < slope < -0.6, \
    f"Empirical slope should be near -1 (got {slope:.3f})"

# --- Figure 5.1: empirical risk vs theoretical bound, log-log axes ---
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.errorbar(n_grid, emp_risk_mean, yerr=emp_risk_se, fmt="o-",
            color=COLOR_LASSO, linewidth=1.8, markersize=7, capsize=3,
            label=r"Empirical $\frac{1}{n_{\mathrm{test}}}\|\mathbf{X}_{\mathrm{test}}(\hat{\boldsymbol{\beta}} - \boldsymbol{\beta}^*)\|^2$")
ax.plot(n_grid, theory_brt, "--", color=COLOR_RIDGE, linewidth=1.5,
        label=fr"BRT bound $72 \sigma^2 s \log(p) / (n \kappa^2)$  (loose)")
ax.plot(n_grid, theory_calibrated, ":", color=COLOR_TRUTH, linewidth=1.5,
        label=fr"Calibrated bound $C \sigma^2 s \log(p)/n$, $C \approx {C_emp:.2f}$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"Sample size $n$ (log scale)")
ax.set_ylabel(r"Prediction risk $\mathrm{PE}(\hat{\boldsymbol{\beta}}^{\mathrm{lasso}})$  (log scale)")
ax.set_title(f"Oracle-inequality rate verification: $p={p_fix}$, $s={s_fix}$, $\\sigma={sigma_fx}$, "
             f"$B={B_oi}$ MC reps")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, which="both", linewidth=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_05_01_oracle_inequality_rate.png", bbox_inches="tight")
plt.show()


## §6. Variable-selection consistency

The §5 oracle inequality bounds the lasso's **prediction risk** — it predicts well, comparable to the oracle that knew $S$. It does **not** say the lasso correctly recovers $S$ itself. These are different statements with different sufficient conditions.

This section formalizes **sign-consistency** (the strongest form of support recovery), introduces the **irrepresentable condition** (Zhao-Yu 2006) that's both sufficient and essentially necessary for it, states the sample-size scaling, and contrasts the prediction-risk bound (RE-based) against the support-recovery bound (IC-based).


### §6.1 Sign-consistency: what it means and why prediction consistency doesn't imply it

Three increasingly strong support-recovery notions:

- **Support recovery:** $\hat A_\lambda := \{j : \hat\beta_j \neq 0\} = S$.
- **Sign consistency:** $\mathrm{sign}(\hat{\boldsymbol\beta}) = \mathrm{sign}(\boldsymbol\beta^*)$ — implies support recovery and gets the active-coefficient signs right.
- **Sign-consistent estimation:** $\mathbb{P}(\mathrm{sign}(\hat{\boldsymbol\beta}^{\text{lasso}}) = \mathrm{sign}(\boldsymbol\beta^*)) \to 1$ as $n \to \infty$.

**Why prediction consistency doesn't imply sign consistency.** Take $\mathbf{X}_1 = \mathbf{X}_2$ (perfectly collinear) with $\boldsymbol\beta^* = (1, 0)$. The lasso objective $\frac{1}{2n}\|\mathbf{y} - \mathbf{X}_1(\beta_1 + \beta_2)\|^2 + \lambda(|\beta_1| + |\beta_2|)$ depends on $(\beta_1, \beta_2)$ only through $\beta_1 + \beta_2$ and $|\beta_1| + |\beta_2|$. Many solutions: $(1, 0), (0, 1), (0.5, 0.5)$, etc. — identical prediction, dramatically different support.

This is extreme but the same phenomenon shows up whenever two features are highly correlated and both predictive — the lasso has no preference between them, may flip across resampled training sets.


### §6.2 The irrepresentable condition (Zhao-Yu 2006)

Given the active set $S$ and sign vector $\mathbf{s}^*_S := \mathrm{sign}(\boldsymbol\beta^*_S)$:

> **Definition 6.1 (irrepresentable condition).** The design $\mathbf{X}$ satisfies the **(weak) irrepresentable condition** for $(S, \mathbf{s}^*_S)$ if
> $$
> \big\| \mathbf{X}_{S^c}^\top \mathbf{X}_S \big(\mathbf{X}_S^\top \mathbf{X}_S\big)^{-1} \mathbf{s}^*_S \big\|_\infty \;\le\; 1.
> $$
> The **strong irrepresentable condition with parameter $\eta > 0$** strengthens this to $\le 1 - \eta$.

**Geometric interpretation.** Let $\mathbf{w}_j := (\mathbf{X}_S^\top \mathbf{X}_S)^{-1} \mathbf{X}_S^\top \mathbf{X}_j$ — the OLS coefficient vector for regressing the inactive feature $\mathbf{X}_j$ ($j \in S^c$) on the active features $\mathbf{X}_S$. Then IC reads $\max_{j \in S^c} |\mathbf{w}_j^\top \mathbf{s}^*_S| \le 1$.

Each $\mathbf{w}_j$ describes how the inactive feature $j$ is predicted by the active features. The IC asks: is the dot product of this prediction recipe with the sign pattern of active coefficients bounded by 1? If not — if some inactive feature is "too aligned" with the active features in a sign-coherent way — the lasso will select it in preference to the true active set.

For random $\mathbf{X}$ with iid rows from $\mathcal{N}(\mathbf{0}, \boldsymbol\Sigma)$, the empirical IC concentrates around the **population IC**: $\|\boldsymbol\Sigma_{S^c, S} \boldsymbol\Sigma_{S, S}^{-1} \mathbf{s}^*_S\|_\infty \le 1$. We compute this for DGP-1 across correlation strengths below.


In [ ]:
# §6.2 — Population irrepresentable quantity for DGP-1-style designs.
# Sigma = AR(1) Toeplitz with parameter rho in [0, 0.95].
# Active set S = {0, ..., 9} contiguous, sign pattern s*_S = (+1,...,+1).
# Compute IC quantity ||Sigma_{S^c, S} Sigma_{S, S}^{-1} s*_S||_inf as a function of rho.

def population_ic(p, s, rho, sign_S):
    """Population irrepresentable quantity for AR(1) Toeplitz Sigma."""
    idx = np.arange(p)
    Sigma = rho ** np.abs(np.subtract.outer(idx, idx))
    S = np.arange(s)
    Sc = np.arange(s, p)
    Sigma_SS  = Sigma[np.ix_(S,  S)]
    Sigma_ScS = Sigma[np.ix_(Sc, S)]
    proj = Sigma_ScS @ np.linalg.solve(Sigma_SS, sign_S)
    return float(np.max(np.abs(proj)))

p_ic   = 500
s_ic   = 10
rho_grid = np.linspace(0.05, 0.95, 30)
ic_pop   = np.array([population_ic(p_ic, s_ic, rho, np.ones(s_ic))
                     for rho in rho_grid])

# Find the rho at which IC crosses 1 (linear interpolation).
above_one = np.where(ic_pop >= 1.0)[0]
if len(above_one) > 0:
    k = above_one[0]
    if k > 0:
        rho_crit = float(np.interp(1.0, [ic_pop[k-1], ic_pop[k]], [rho_grid[k-1], rho_grid[k]]))
    else:
        rho_crit = float(rho_grid[k])
else:
    rho_crit = float("nan")
print(f"DGP-1-style population IC crosses 1 at approximately rho ~ {rho_crit:.3f}")
print(f"  IC at rho=0.5 (DGP-1 default): {population_ic(p_ic, s_ic, 0.5, np.ones(s_ic)):.4f}")
print(f"  IC at rho=0.7:                {population_ic(p_ic, s_ic, 0.7, np.ones(s_ic)):.4f}")
print(f"  IC at rho=0.9:                {population_ic(p_ic, s_ic, 0.9, np.ones(s_ic)):.4f}")

# Sanity: IC should be monotonically increasing in rho for this DGP.
assert np.all(np.diff(ic_pop) > -1e-6), \
    "IC should be monotonically nondecreasing in rho for AR(1) Toeplitz Sigma"

# --- Figure 6.1 ---
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.plot(rho_grid, ic_pop, "o-", color=COLOR_LASSO, linewidth=1.8, markersize=6,
        label=r"Population IC $\|\boldsymbol{\Sigma}_{S^c, S} \boldsymbol{\Sigma}_{S,S}^{-1} \mathbf{s}^*_S\|_\infty$")
ax.axhline(1.0, color=COLOR_DEBIASED, linewidth=1.5, linestyle="--",
           label=r"IC threshold = 1 (lasso loses sign-consistency above)")
ax.axvline(0.5, color="#888888", linewidth=0.8, linestyle=":",
           label=r"$\rho = 0.5$ (DGP-1 default)")
if not np.isnan(rho_crit):
    ax.axvline(rho_crit, color=COLOR_RIDGE, linewidth=1.0, linestyle=":",
               label=fr"crossover $\rho^* \approx {rho_crit:.2f}$")
# Shade the "lasso sign-consistent" region.
ax.fill_between(rho_grid, 0, ic_pop, where=(ic_pop <= 1.0),
                color=COLOR_LASSO, alpha=0.10, label="IC < 1: lasso sign-consistent")

ax.set_xlabel(r"Correlation strength $\rho$ in AR(1) $\boldsymbol{\Sigma}_{jk} = \rho^{|j-k|}$")
ax.set_ylabel("Irrepresentable quantity")
ax.set_title(f"Irrepresentable condition for DGP-1-style designs: $p = {p_ic}$, $s = {s_ic}$, "
             r"$S = \{0,\dots,9\}$, $\mathbf{s}^*_S = \mathbf{1}$")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, linewidth=0.3, alpha=0.4)
ax.set_ylim(0, max(1.5, ic_pop.max() * 1.05))
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_06_01_irrepresentable_condition.png", bbox_inches="tight")
plt.show()


### §6.3 The sample-size scaling for support recovery

> **Theorem 6.1 (lasso sign-consistency; Zhao-Yu 2006; Wainwright 2009).** Suppose the design satisfies strong IC with parameter $\eta > 0$, the active-set Gram matrix is well-conditioned ($\lambda_{\min}(\mathbf{X}_S^\top \mathbf{X}_S / n) \ge C_{\min}$), the columns are normalized ($\|\mathbf{X}_j\|^2 \le n$), the minimum signal is large enough ($\beta^*_{\min} \gtrsim \lambda$), and the noise is sub-Gaussian. Choosing $\lambda \ge \frac{2\sigma}{\eta}\sqrt{2\log(p)/n}$ and $n \gtrsim (s/C_{\min}) \log(p)$:
> $$
> \mathbb{P}\big(\mathrm{sign}(\hat{\boldsymbol\beta}^{\text{lasso}}) = \mathrm{sign}(\boldsymbol\beta^*)\big) \to 1.
> $$

**Proof sketch (primal-dual witness).** Wainwright's (2009) proof has five steps:

1. **Restricted lasso.** Solve the lasso *only* on the active features: $\tilde{\boldsymbol\beta}_S = \arg\min \frac{1}{2n}\|\mathbf{y} - \mathbf{X}_S \boldsymbol\beta_S\|^2 + \lambda\|\boldsymbol\beta_S\|_1$. Set $\tilde{\boldsymbol\beta}_{S^c} = \mathbf{0}$.
2. **Sign verification.** Verify $\mathrm{sign}(\tilde{\boldsymbol\beta}_S) = \mathrm{sign}(\boldsymbol\beta^*_S)$ — uses the minimum-signal condition.
3. **Construct the dual.** Set $\tilde{\mathbf{g}}_S = \mathrm{sign}(\tilde{\boldsymbol\beta}_S)$, derive $\tilde{\mathbf{g}}_{S^c}$ from the active KKT condition.
4. **Verify $\|\tilde{\mathbf{g}}_{S^c}\|_\infty < 1$.** The leading term is the irrepresentable quantity (bounded by $1 - \eta$ via strong IC); the noise term is $O(\sigma\sqrt{\log(p)/n}) \ll \eta$ for large $n$.
5. **Conclude.** By KKT uniqueness, $\tilde{\boldsymbol\beta} = \hat{\boldsymbol\beta}^{\text{lasso}}$. Sign consistency follows from steps 1–2. $\blacksquare$

**Necessity of IC.** Wainwright (2009, Theorem 3) also proved the converse: if IC fails ($\|\cdot\|_\infty > 1 + \delta$), then $\mathbb{P}(\mathrm{sign}(\hat{\boldsymbol\beta}) = \mathrm{sign}(\boldsymbol\beta^*)) \to 0$ regardless of $\lambda$. So IC isn't a proof artifact — it's the *correct* characterization of when the lasso can do support recovery.


In [ ]:
# §6.3 — Empirical demonstration: lasso support recovery succeeds at low rho
# (IC < 1) and fails at high rho (IC > 1).

from sklearn.linear_model import LassoCV as _LassoCV

def empirical_support_recovery(rho, n_train=400, p_demo=200, s_demo=10,
                                B_demo=40, sigma_demo=0.3):
    """Estimate P(supp(beta_hat_lasso) = S) and the false-positive count at rho."""
    matches, fp_counts, tp_counts = 0, [], []
    for b in range(B_demo):
        rng_b = np.random.default_rng(SEED + 8000 + 100*int(rho*100) + b)
        X_b, y_b, beta_b, _ = make_dgp1(
            n=n_train, p=p_demo, s=s_demo, sigma=sigma_demo, rho=rho, rng=rng_b)
        S_true = np.abs(beta_b) > 0
        # Use LassoCV for a practical lambda choice.
        fit = _LassoCV(cv=5, fit_intercept=False, alphas=30,
                       max_iter=20_000, random_state=SEED + b).fit(X_b, y_b)
        S_hat = np.abs(fit.coef_) > 1e-6
        if np.array_equal(S_true, S_hat):
            matches += 1
        # True positives, false positives.
        tp_counts.append(int(np.sum(S_hat & S_true)))
        fp_counts.append(int(np.sum(S_hat & ~S_true)))
    return matches / B_demo, np.mean(tp_counts), np.mean(fp_counts)

print("Empirical support-recovery rates at LassoCV-selected lambda "
      "(n=400, p=200, s=10, sigma=0.3, B=40 reps):")
print(f"{'rho':>5}  {'pop IC':>8}  {'P(exact recovery)':>20}  {'avg TP':>8}  {'avg FP':>8}")
for rho in [0.2, 0.5, 0.7, 0.85, 0.95]:
    pop_ic_rho = population_ic(200, 10, rho, np.ones(10))
    rate, tp, fp = empirical_support_recovery(rho)
    print(f"{rho:>5.2f}  {pop_ic_rho:>8.3f}  {rate:>20.2f}  {tp:>8.2f}  {fp:>8.2f}")

# Sanity: support recovery should be much better at rho=0.2 than rho=0.95.
rate_low, _, _  = empirical_support_recovery(0.2)
rate_high, _, _ = empirical_support_recovery(0.95)
print(f"\nSanity check: P(exact recovery) at rho=0.2 ({rate_low:.2f}) vs rho=0.95 ({rate_high:.2f})")


### §6.4 Contrasting prediction-risk and support-recovery

The two main theorems of §§5–6 differ on every axis:

| | Prediction-risk bound (§5) | Support-recovery bound (§6) |
|---|---|---|
| **What's bounded** | $\frac{1}{n}\|\mathbf{X}(\hat{\boldsymbol\beta} - \boldsymbol\beta^*)\|_2^2$ | $\mathbb{P}(\mathrm{sign}(\hat{\boldsymbol\beta}) = \mathrm{sign}(\boldsymbol\beta^*))$ |
| **Sufficient design condition** | Restricted-eigenvalue (§5.4) | Irrepresentable (§6.2) |
| **Necessary?** | RE essentially necessary at the optimal rate | Strong IC necessary for *lasso* sign-consistency |
| **Sample-size scaling** | $n \gtrsim s \log(p)$ | $n \gtrsim s \log(p)$ |
| **Minimum-signal needed?** | No | Yes — $\beta^*_{\min} \gtrsim \lambda$ required |
| **Failure mode** | Larger $n$ helps (better RE) | Lasso fundamentally can't recover; need adaptive lasso (§8.3) or post-selection refit |

**The conditions are not nested.** RE doesn't imply IC, and IC doesn't imply RE. They measure different geometric properties: RE is a lower bound on $\mathbf{X}^\top \mathbf{X}/n$ restricted to a cone (a global property, doesn't depend on which $S$ is the active set); IC depends on $S$ and $\mathbf{s}^*_S$ specifically.

**Practical implications.** The lasso is a much better prediction tool than a model-selection tool.

- **For prediction:** trust the lasso. CV-selected $\lambda$, refit at $\lambda_{\text{CV}}$ or $\lambda_{1\mathrm{SE}}$. The §5 oracle inequality gives near-optimal prediction risk under mild conditions.
- **For variable selection:** be skeptical of the lasso's chosen support. Watch for (i) two highly correlated features where only one shows up — the lasso may have arbitrarily picked one — and (ii) the lasso fit changing dramatically across resampled training sets — instability suggests IC is violated. Use stability selection (Meinshausen-Bühlmann 2010) or the **adaptive lasso** (§8.3) for sign-consistency under weaker conditions.

The bridge to §§7–10: practical $\lambda$-selection (§7) trades off these objectives differently — `LassoCV` optimizes prediction (smaller $\lambda$, more features); `LassoLarsIC`/BIC penalizes model size more heavily. The debiased lasso (§10) sidesteps the support-recovery question entirely by producing valid CIs for individual coefficients without requiring sign consistency.


## §7. Cross-validation for $\lambda$

The §5 oracle inequality recommends $\lambda \asymp \sigma \sqrt{\log(p)/n}$ — a rate, not a constant. The constant matters in practice, and the noise scale $\sigma$ is rarely known. Practical $\lambda$-selection uses data-driven criteria: **cross-validation** (the default), the **one-standard-error rule** (a parsimony-favoring variant), and **information criteria** like AIC/BIC computed along the lasso path. This section covers all three.

The CV / 1-SE / BIC distinction maps onto the §6 discussion: CV optimizes prediction error (and tends to over-select); BIC penalizes model size more aggressively (and is sometimes selection-consistent); the 1-SE rule is a Hastie-Tibshirani-Friedman compromise. None is "right" — the right choice depends on whether you care about prediction or interpretability.


### §7.1 $K$-fold cross-validation with `LassoCV`

$K$-fold CV estimates the prediction risk at each $\lambda$ by holdout:

1. Partition training data into $K$ folds.
2. For each fold $k$ and each $\lambda$, fit the lasso on the data minus fold $k$, compute held-out MSE on fold $k$.
3. Average: $\mathrm{CV}(\lambda) = (1/K) \sum_k \mathrm{MSE}_k(\lambda)$.
4. Select $\lambda_{\min} = \arg\min_\lambda \mathrm{CV}(\lambda)$.

**Standard choices:** $K = 10$ for general use, $K = 5$ if compute is constrained, leave-one-out only when $n$ is small. The $\lambda$-grid is typically log-spaced from $\lambda_{\max}$ down to $\lambda_{\max} \cdot 10^{-3}$ with ~100 grid points.

**Computational efficiency.** Naively $K \times |\Lambda|$ lasso fits, but warm starts along the $\lambda$ path (recall §3.2) reduce this to ~$K$ path computations. For DGP-1 with $K=10$, $|\Lambda|=100$, $p=500$: ~1–3 seconds total.


### §7.2 The one-standard-error rule

The CV estimate is itself random. The **fold-wise standard error** $\mathrm{SE}(\lambda) := \mathrm{sd}_k(\mathrm{MSE}_k(\lambda)) / \sqrt{K}$ quantifies its uncertainty. The **one-standard-error rule** (Breiman et al. 1984; ESL §7.10) selects the most parsimonious model whose CV error is within one SE of the minimum:

$$
\lambda_{1\mathrm{SE}} := \max\{\lambda : \mathrm{CV}(\lambda) \le \mathrm{CV}(\lambda_{\min}) + \mathrm{SE}(\lambda_{\min})\}.
$$

Since $\mathrm{CV}(\lambda)$ is roughly U-shaped, $\lambda_{1\mathrm{SE}} > \lambda_{\min}$ — the 1-SE model is more regularized, sparser.

**Why use it.** $\lambda_{\min}$ is unbiased for MSE but unstable across resampled training data — a small perturbation can shift $\lambda_{\min}$ by a factor of 2. The 1-SE rule trades this instability for a small, controlled increase in prediction error: the resulting model has CV-MSE indistinguishable from $\lambda_{\min}$'s but is more parsimonious and reproducible. Typical lasso behavior: $|\hat A_{1\mathrm{SE}}|$ is 10–30% smaller than $|\hat A_{\min}|$, with test MSE 5–15% larger.

**Caveat.** The 1-SE rule is a heuristic, not a theorem — no provable support consistency, no provable prediction-risk improvement. If you need either, use BIC (§7.3) or the §5 oracle inequality directly.


In [ ]:
# §7.2 — Figure 7.1: LassoCV curve with lambda_min and lambda_1SE markers.
# Use the §1 sample. Reuse the LassoCV fit from §1.4 (lasso_cv) — it already has
# the path stored in mse_path_, alphas_.

# (Re-fit here with cv=10 and store internals for the figure.)
fit_cv_10 = LassoCV(cv=10, fit_intercept=False, alphas=100,
                     random_state=SEED, max_iter=20_000).fit(X1, y1)
alphas_cv = fit_cv_10.alphas_                           # decreasing
mse_path  = fit_cv_10.mse_path_                          # (n_alphas, n_folds)
mse_mean  = mse_path.mean(axis=1)
mse_se    = mse_path.std(axis=1, ddof=1) / np.sqrt(mse_path.shape[1])

# lambda_min (= fit_cv_10.alpha_).
idx_min = int(np.argmin(mse_mean))
lam_min_cv = float(alphas_cv[idx_min])

# lambda_1SE: largest lambda with mse_mean <= mse_mean[idx_min] + mse_se[idx_min].
# Since alphas_cv is decreasing, "larger lambda" = smaller index.
threshold = mse_mean[idx_min] + mse_se[idx_min]
candidates = np.where(mse_mean[:idx_min + 1] <= threshold)[0]
idx_1se = int(candidates[0])
lam_1se = float(alphas_cv[idx_1se])

print(f"LassoCV (10-fold) on §1 DGP:")
print(f"  lambda_min  = {lam_min_cv:.4f}  (CV MSE = {mse_mean[idx_min]:.4f} +/- {mse_se[idx_min]:.4f})")
print(f"  lambda_1SE  = {lam_1se:.4f}  (CV MSE = {mse_mean[idx_1se]:.4f})")
print(f"  ratio lambda_1SE / lambda_min = {lam_1se / lam_min_cv:.2f}")

# Sanity: lambda_1SE >= lambda_min, and CV MSE at lambda_1SE <= threshold.
assert lam_1se >= lam_min_cv, "1-SE lambda should be >= minimum-CV lambda"
assert mse_mean[idx_1se] <= threshold + 1e-10, "1-SE lambda should satisfy threshold"

# --- Figure 7.1 ---
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.plot(alphas_cv, mse_mean, "o-", color=COLOR_LASSO, linewidth=1.4, markersize=4,
        label="Mean CV MSE (10 folds)")
ax.fill_between(alphas_cv, mse_mean - mse_se, mse_mean + mse_se,
                color=COLOR_LASSO, alpha=0.18, label=r"$\pm 1$ SE band")
ax.axvline(lam_min_cv, color=COLOR_RIDGE, linewidth=1.5, linestyle="--",
           label=fr"$\lambda_{{\min}} = {lam_min_cv:.4f}$")
ax.axvline(lam_1se, color=COLOR_DEBIASED, linewidth=1.5, linestyle="--",
           label=fr"$\lambda_{{1\mathrm{{SE}}}} = {lam_1se:.4f}$")
ax.axhline(threshold, color="#888888", linewidth=0.7, linestyle=":",
           alpha=0.7, label=r"CV($\lambda_{\min}$) + 1 SE")
ax.set_xscale("log")
ax.set_xlabel(r"$\lambda$ (log scale)")
ax.set_ylabel("Mean CV MSE")
ax.set_title(r"LassoCV (10-fold) on §1 DGP --- $\lambda_{\min}$ vs $\lambda_{1\mathrm{SE}}$")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, which="both", linewidth=0.3, alpha=0.4)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_07_01_cv_curve.png", bbox_inches="tight")
plt.show()


### §7.3 BIC selection with `LassoLarsIC`

Information criteria balance model fit against complexity through an explicit complexity penalty. For a candidate $\lambda$ with active-set size $k_\lambda := |\hat A_\lambda|$ and residual sum of squares $\mathrm{RSS}_\lambda$:

$$
\mathrm{AIC}(\lambda) = n \log(\mathrm{RSS}_\lambda / n) + 2 k_\lambda,
$$

$$
\mathrm{BIC}(\lambda) = n \log(\mathrm{RSS}_\lambda / n) + k_\lambda \log n.
$$

Both penalize larger models; BIC penalizes more aggressively for $n > e^2 \approx 7.4$, so BIC selects smaller models.

Using $k_\lambda = |\hat A_\lambda|$ as the lasso's effective dof rests on **Zou-Hastie-Tibshirani (2007)**: the size of the active set is an unbiased estimator of the lasso's degrees of freedom, $\mathbb{E}[k_\lambda] = \mathrm{df}(\hat{\boldsymbol\beta}^{\text{lasso}}(\lambda))$. This is non-trivial; for a generic non-linear estimator, dof is not the count of nonzero parameters.

**`LassoLarsIC`.** Computes the lasso path via LARS (Efron-Hastie-Johnstone-Tibshirani 2004) — exploits piecewise linearity to enumerate every knot in $O(np \min(n, p))$. IC values computed at each knot; the minimizing $\lambda$ is returned. Path-based and exact (no $\lambda$-grid discretization), but only practical for moderate $p$.

**Selection consistency.** BIC is **selection-consistent** under additional conditions: $\mathbb{P}(\hat A_{\lambda_{\mathrm{BIC}}} = S) \to 1$ as $n \to \infty$ given a slightly stronger condition than IC and a bounded-below minimum signal (Wang-Li-Tsai 2009). AIC and CV are not — they tend to over-select even asymptotically. For variable-selection applications, this is BIC's main virtue.


### §7.4 Comparison on the §1 toy DGP

The four selectors make different trade-offs on the same data: `LassoCV` $\lambda_{\min}$ minimizes prediction error, `LassoCV` $\lambda_{1\mathrm{SE}}$ trades it for parsimony, `LassoLarsIC` BIC trades it more aggressively for selection consistency, and `LassoLarsIC` AIC sits between the two CV variants.

**Recommendation:**

- **For prediction:** `LassoCV` with $\lambda_{\min}$.
- **For prediction with a smaller, more interpretable model:** the 1-SE rule.
- **For selection consistency:** BIC via `LassoLarsIC(criterion='bic')`.
- **As a default:** start with `LassoCV` $\lambda_{\min}$ — the universal lasso default.

The §10 debiased lasso uses `LassoCV` $\lambda_{\min}$ as its initial estimator. The choice of $\lambda$ isn't critical for debiased-lasso CI coverage — the one-step correction substantially compensates for the lasso's selection idiosyncrasies.


In [ ]:
# §7.4 — Compare four selectors on §1 DGP. Single fit (no MC) for clarity.
# IC-based selectors are computed manually along the §4.4 lasso path, replacing
# the LassoLarsIC selector class — sklearn 1.8+ refuses LassoLarsIC for n < p
# because the unbiased noise-variance estimator (RSS / (n - df)) breaks down
# when the LARS path can drive RSS to ~0. Computing BIC/AIC directly on the
# warm-started coordinate-descent path side-steps that pathology while
# preserving the §7.4 lesson: IC-based selection vs CV-based selection.

# Fresh test set for evaluation.
X_test_71, _, beta_star_71, _ = make_dgp1(
    n=2000, rng=np.random.default_rng(SEED + 9000))

# --- IC-based selection along the §4.4 lasso path ---
# At each lambda, k_lambda = active-set size, RSS_lambda = ||y - X beta_hat||^2.
# BIC_lambda = n log(RSS/n) + k log(n)
# AIC_lambda = n log(RSS/n) + 2k
# For n < p the lasso fit at very small lambda can drive RSS toward 0; we floor
# RSS away from zero so log(RSS/n) stays finite. The floor is well below any
# meaningful IC value at sane lambda.
n_71 = X1.shape[0]
RSS_path = np.array([float(np.sum((y1 - X1 @ path_coefs[:, k])**2))
                     for k in range(path_coefs.shape[1])])
k_path   = np.array([int(np.sum(np.abs(path_coefs[:, k]) > 1e-10))
                     for k in range(path_coefs.shape[1])])
RSS_clip = np.maximum(RSS_path, 1e-12)
bic_path = n_71 * np.log(RSS_clip / n_71) + k_path * np.log(n_71)
aic_path = n_71 * np.log(RSS_clip / n_71) + 2 * k_path

idx_bic = int(np.argmin(bic_path))
idx_aic = int(np.argmin(aic_path))
lam_bic = float(lambda_path[idx_bic])
lam_aic = float(lambda_path[idx_aic])

# --- Four methods, all refit at the selected lambda for an apples-to-apples
# comparison of the resulting beta_hat (path coefs use warm starts and may
# have looser tol than a from-scratch fit). ---

# 1. LassoCV lambda_min  (computed in §7.2 as fit_cv_10, lam_min_cv)
beta_cv_min = Lasso(alpha=lam_min_cv, fit_intercept=False,
                    max_iter=20_000).fit(X1, y1).coef_

# 2. LassoCV lambda_1SE  (computed in §7.2 as lam_1se)
beta_cv_1se = Lasso(alpha=lam_1se, fit_intercept=False,
                    max_iter=20_000).fit(X1, y1).coef_

# 3. Manual BIC along the lasso path
beta_bic = Lasso(alpha=lam_bic, fit_intercept=False,
                 max_iter=20_000).fit(X1, y1).coef_

# 4. Manual AIC along the lasso path
beta_aic = Lasso(alpha=lam_aic, fit_intercept=False,
                 max_iter=20_000).fit(X1, y1).coef_

methods = [
    ("LassoCV (lambda_min)",   beta_cv_min, lam_min_cv, COLOR_LASSO),
    ("LassoCV (lambda_1SE)",   beta_cv_1se, lam_1se,    COLOR_DEBIASED),
    ("Lasso path + BIC",       beta_bic,    lam_bic,    COLOR_RIDGE),
    ("Lasso path + AIC",       beta_aic,    lam_aic,    COLOR_OLS),
]

print(f"{'Method':<24}  {'lambda':>10}  {'|A_hat|':>8}  {'|A cap S|':>10}  {'test MSE':>10}")
results = []
for name, beta_hat, lam_used, _ in methods:
    n_active = int(np.sum(np.abs(beta_hat) > 1e-10))
    n_in_S   = int(np.sum(np.abs(beta_hat[:10]) > 1e-10))
    test_mse = float(np.mean((X_test_71 @ (beta_hat - beta_star_71)) ** 2))
    print(f"{name:<24}  {lam_used:>10.4f}  {n_active:>8d}  "
          f"{n_in_S:>7d}/10  {test_mse:>10.4f}")
    results.append((name, lam_used, n_active, n_in_S, test_mse))

# Sanity: lambda_BIC >= lambda_AIC (BIC penalty harsher) and
# lambda_1SE >= lambda_min (1-SE rule more parsimonious).
assert lam_bic >= lam_aic - 1e-6, "BIC should select lambda >= AIC's lambda (BIC penalty harsher)"
assert lam_1se >= lam_min_cv - 1e-6, "1-SE rule should select lambda >= CV-min's lambda"

# --- Figure 7.2: side-by-side bar chart of |A_hat| and test MSE ---
method_names = [r[0] for r in results]
n_actives    = [r[2] for r in results]
test_mses    = [r[4] for r in results]
colors       = [m[3] for m in methods]

def two_line_label(name):
    """Wrap selector labels for readable x-tick text."""
    return name.replace(" (", "\n(").replace(" + ", "\n+ ")

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.5))

# Active set size.
ax = axes[0]
bars = ax.bar(range(len(method_names)), n_actives, color=colors, alpha=0.85)
ax.axhline(10, color="black", linewidth=0.8, linestyle=":",
           label=r"true sparsity $s = 10$")
ax.set_xticks(range(len(method_names)))
ax.set_xticklabels([two_line_label(m) for m in method_names], fontsize=9)
ax.set_ylabel(r"Active set size $|\hat A_\lambda|$")
ax.set_title("Model size by selector")
for bar, val in zip(bars, n_actives):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.4, str(val),
            ha="center", fontsize=10)
ax.legend(loc="upper right", fontsize=9)

# Test MSE.
ax = axes[1]
bars = ax.bar(range(len(method_names)), test_mses, color=colors, alpha=0.85)
ax.set_xticks(range(len(method_names)))
ax.set_xticklabels([two_line_label(m) for m in method_names], fontsize=9)
ax.set_ylabel("Prediction MSE on test set")
ax.set_title("Prediction performance by selector")
for bar, val in zip(bars, test_mses):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + max(test_mses) * 0.01, f"{val:.3f}",
            ha="center", fontsize=10)

fig.suptitle(f"Selector comparison on §1 DGP "
             f"($n={X1.shape[0]}$, $p={X1.shape[1]}$, true $s=10$)", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_07_02_selector_comparison.png", bbox_inches="tight")
plt.show()


## §8. Ridge, elastic net, and adaptive lasso

The lasso has three practical limitations: it's unstable on highly correlated features (§6.1), biases active coefficients by $\lambda$ (§4.1), and requires the irrepresentable condition for support recovery (§6.2). Three variants address these:

- **Ridge** — L2 penalty, dense unique solution, stable on correlated features, but no selection.
- **Elastic net** (Zou-Hastie 2005) — L1 + L2, lasso-style sparsity with ridge-style grouping of correlated features.
- **Adaptive lasso** (Zou 2006) — feature-specific weights, achieves the oracle property under weaker conditions than IC.

Decision tree: dense truth → ridge; sparse truth with separated features → lasso; sparse truth with correlated groups → elastic net; sparse truth, need unbiased coefficients → adaptive lasso.


### §8.1 Ridge: continuous shrinkage, no selection

Recall from §1.3: $\hat{\boldsymbol\beta}^{\text{ridge}}(\lambda) = (\mathbf{X}^\top \mathbf{X} + n\lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}$. The matrix $\mathbf{X}^\top \mathbf{X} + n\lambda \mathbf{I}$ is positive definite for any $\lambda > 0$ regardless of $p$ vs $n$ — ridge has no failure mode.

In the SVD basis $\mathbf{X} = \mathbf{U} \mathbf{D} \mathbf{V}^\top$, ridge shrinks the $j$-th SVD coefficient by $d_j^2 / (d_j^2 + n\lambda)$. Small singular values (noisy directions) get heavy shrinkage; large singular values (signal directions) get light shrinkage. But no coefficient is zeroed out — the solution is generically dense.

**When ridge wins.** Truly dense $\boldsymbol\beta^*$, or heavy multicollinearity with no sparsity prior. **When ridge loses.** When $\boldsymbol\beta^*$ is sparse, the cumulative variance contribution from the 490 inactive (but nonzero) coordinates is substantial. On DGP-1, lasso prediction MSE is typically 2–5× better than ridge.

For most high-dimensional ML problems the truth is sparse, so the lasso wins. The standard practice is to compare both via cross-validated test MSE.


### §8.2 The elastic net for groups of correlated features

The elastic net (Zou-Hastie 2005) combines L1 + L2:

$$
\hat{\boldsymbol\beta}^{\text{EN}}(\lambda, \alpha) = \arg\min_{\boldsymbol\beta} \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|^2 + \lambda\left[\alpha \|\boldsymbol\beta\|_1 + \frac{1-\alpha}{2}\|\boldsymbol\beta\|_2^2\right],
$$

with mixing $\alpha \in [0, 1]$. $\alpha = 1$ is pure lasso, $\alpha = 0$ is pure ridge. Standard: $\alpha = 0.5$ or CV-tuned.

**Two structural advantages over lasso:**
1. **Strict convexity for $\alpha < 1$.** The L2 term makes the objective strictly convex in $\boldsymbol\beta$, so the solution is unique even with duplicate columns. The §6.1 collinear-features pathology disappears.
2. **Grouping effect.** For correlated $\mathbf{X}_j, \mathbf{X}_k$, Zou-Hastie 2005 Theorem 1: $|\hat\beta_j^{\text{EN}} - \hat\beta_k^{\text{EN}}| \le \frac{\|\mathbf{y}\|_2}{n\lambda(1-\alpha)}\sqrt{2(1-\rho_{jk})}$. As $\rho_{jk} \to 1$, perfectly correlated features get equal coefficients.

**Reduction to standard lasso.** Augment $\tilde{\mathbf{X}} = \begin{pmatrix} \mathbf{X} \\ \sqrt{n\lambda(1-\alpha)} \mathbf{I}_p \end{pmatrix}$, $\tilde{\mathbf{y}} = \begin{pmatrix} \mathbf{y} \\ \mathbf{0}_p \end{pmatrix}$, run lasso at $\lambda\alpha$. All lasso solvers transfer to the elastic net by this augmentation.

**When elastic net wins.** Highly correlated *groups* of features. Genomics (SNPs in linkage disequilibrium) is canonical. **When it loses.** Well-separated features — L2 adds small bias without reducing variance much.


### §8.3 The adaptive lasso and the oracle property

The vanilla lasso biases every active coefficient by $-\lambda \mathrm{sign}(\beta^*_j)$ regardless of $|\beta^*_j|$. The **adaptive lasso** (Zou 2006) replaces the uniform L1 penalty with feature-specific weights:

$$
\hat{\boldsymbol\beta}^{\text{aLasso}}(\lambda) = \arg\min_{\boldsymbol\beta} \frac{1}{2n}\|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|^2 + \lambda \sum_j w_j |\beta_j|, \qquad w_j = \frac{1}{|\hat\beta_j^{\text{init}}|^\gamma},
$$

with initial estimator $\hat{\boldsymbol\beta}^{\text{init}}$ (typically LassoCV when $p \ge n$, OLS otherwise) and $\gamma > 0$ (standard $\gamma = 1$). Coordinates with large initial estimates get small weights (light shrinkage); near-zero initial estimates get large weights (heavy shrinkage, effectively forced to zero).

**Reduction to standard lasso.** Rescale $\tilde{\mathbf{X}}_j = \mathbf{X}_j / w_j$, $\tilde\beta_j = w_j \beta_j$. The weighted lasso becomes a standard lasso on $(\tilde{\mathbf{X}}, \mathbf{y})$; unscale after fitting.

> **Theorem 8.1 (oracle property; Zou 2006).** Under $\sqrt{n}$-consistency of $\hat{\boldsymbol\beta}^{\text{init}}$, minimum-signal condition, sub-Gaussian noise, and $\lambda_n$ satisfying $\sqrt{n}\lambda_n \to 0$ and $\sqrt{n}(n\lambda_n)^{\gamma/2 - 1} \to \infty$, the adaptive lasso satisfies:
>
> (i) **Selection consistency:** $\mathbb{P}(\hat A^{\text{aLasso}}_{\lambda_n} = S) \to 1$.
>
> (ii) **Asymptotic normality:** $\sqrt{n}(\hat{\boldsymbol\beta}_S^{\text{aLasso}} - \boldsymbol\beta^*_S) \overset{d}{\to} \mathcal{N}(\mathbf{0}, \sigma^2 (\boldsymbol\Sigma_{S, S})^{-1})$ — the same as the oracle OLS-on-$S$ estimator.

The oracle property is stronger than sign-consistency: in addition to recovering $S$, the active-coefficient estimates have *correct* asymptotic standard errors. The constant shrinkage bias disappears asymptotically because $w_j = 1/|\hat\beta_j^{\text{init}}|$ stays bounded for active coords but diverges for inactive.

**Conditions weaker than IC.** Adaptive lasso achieves selection consistency under any design with a $\sqrt{n}$-consistent initial — strictly weaker than strong IC. Designs where vanilla lasso fails support recovery can still admit adaptive-lasso recovery.


In [ ]:
# §8.4 — Three-panel solution path comparison: ridge, lasso, elastic net.

from sklearn.linear_model import enet_path

# Common lambda-grid for the three paths. Use a logspace anchored to lambda_max from §4.
lambda_path_8 = np.logspace(np.log10(lambda_max * 1.01),
                             np.log10(lambda_max * 5e-3), 60)

# 1. Ridge path: closed form at each lambda.
ridge_coefs = np.zeros((X1.shape[1], len(lambda_path_8)))
for k, lam in enumerate(lambda_path_8):
    ridge_coefs[:, k] = Ridge(alpha=lam * X1.shape[0],
                              fit_intercept=False).fit(X1, y1).coef_

# 2. Lasso path (reuse the §4 logic).
_, lasso_coefs_path, _ = lasso_path(X1, y1, alphas=lambda_path_8,
                                     tol=1e-7, max_iter=20_000)

# 3. Elastic net path at alpha=0.5.
_, enet_coefs_path, _ = enet_path(X1, y1, alphas=lambda_path_8, l1_ratio=0.5,
                                   tol=1e-7, max_iter=20_000)

# --- Figure 8.1: three-panel solution-path comparison ---
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8), sharey=True)

for ax, coefs, title, color in [
    (axes[0], ridge_coefs,        "Ridge (L2)",                COLOR_RIDGE),
    (axes[1], lasso_coefs_path,   "Lasso (L1)",                COLOR_LASSO),
    (axes[2], enet_coefs_path,    r"Elastic net ($\alpha=0.5$)", COLOR_ENET),
]:
    # Plot inactive coords (j >= 10) thin and gray.
    for j in range(10, X1.shape[1]):
        ax.plot(lambda_path_8, coefs[j, :], color=COLOR_OLS,
                alpha=0.25, linewidth=0.4)
    # Plot active coords (j < 10) thick and black.
    for j in range(10):
        ax.plot(lambda_path_8, coefs[j, :], color=COLOR_TRUTH,
                alpha=0.95, linewidth=1.6)
    ax.axhline(1.0, color="#aaaaaa", linewidth=0.7, linestyle=":",
               alpha=0.7, label=r"$\beta^*_j = 1$" if title.startswith("Ridge") else None)
    ax.axhline(0, color="black", linewidth=0.4)
    ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda$ (log scale)")
    ax.set_title(title, color=color)
    ax.invert_xaxis()
    if title.startswith("Ridge"):
        ax.legend(loc="upper left", fontsize=9)

axes[0].set_ylabel(r"$\hat\beta_j(\lambda)$")
fig.suptitle("Solution paths on §1 DGP: true active $j \\in S$ in black, "
             "noise coords in gray", y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_08_01_ridge_lasso_enet_paths.png", bbox_inches="tight")
plt.show()


In [ ]:
# §8.3 — Adaptive lasso vs vanilla lasso: bias reduction on active coordinates.

def adaptive_lasso(X, y, gamma=1.0, cv=5, eps=1e-4):
    """Adaptive lasso (Zou 2006).
    Step 1: Initial estimator via LassoCV.
    Step 2: Weights w_j = 1 / (|beta_init,j| + eps)^gamma.
    Step 3: Weighted lasso via feature rescaling, re-tuned by CV.
    """
    # Initial estimator: LassoCV.
    init_fit = LassoCV(cv=cv, fit_intercept=False, alphas=50,
                       max_iter=20_000, random_state=SEED).fit(X, y)
    beta_init = init_fit.coef_

    # Weights (with floor eps to avoid division by zero / numerical issues).
    w = 1.0 / (np.abs(beta_init) + eps) ** gamma

    # Rescale features and re-fit LassoCV.
    X_tilde = X / w[None, :]
    adapt_fit = LassoCV(cv=cv, fit_intercept=False, alphas=50,
                        max_iter=20_000, random_state=SEED).fit(X_tilde, y)
    beta_tilde = adapt_fit.coef_

    # Unscale.
    beta_hat = beta_tilde / w
    return beta_hat, float(adapt_fit.alpha_), beta_init

beta_adapt, lam_adapt, beta_init_for_adapt = adaptive_lasso(X1, y1, gamma=1.0)

# Compare to vanilla lasso at LassoCV lambda_min (computed in §7.4).
print(f"Adaptive lasso vs vanilla lasso on §1 DGP active set "
      f"S = {{0, ..., 9}}, true beta*_j = 1:")
print(f"  Vanilla lasso (LassoCV lambda_min = {lam_min_cv:.4f}):")
print(f"    mean |beta_S|         = {np.mean(np.abs(beta_cv_min[:10])):.4f}  (truth: 1.000)")
print(f"    bias on S          = {np.mean(beta_cv_min[:10] - 1.0):+.4f}")
print(f"    |A_hat|                = {int(np.sum(np.abs(beta_cv_min) > 1e-10))}")
print(f"  Adaptive lasso (LassoCV lambda = {lam_adapt:.4f}, gamma = 1):")
print(f"    mean |beta_S|         = {np.mean(np.abs(beta_adapt[:10])):.4f}  (truth: 1.000)")
print(f"    bias on S          = {np.mean(beta_adapt[:10] - 1.0):+.4f}")
print(f"    |A_hat|                = {int(np.sum(np.abs(beta_adapt) > 1e-10))}")

# Sanity: adaptive lasso should produce less-shrunk active coefficients.
mean_active_vanilla = float(np.mean(np.abs(beta_cv_min[:10])))
mean_active_adapt   = float(np.mean(np.abs(beta_adapt[:10])))
assert mean_active_adapt > mean_active_vanilla - 1e-3, \
    "Adaptive lasso should produce active coefficients at least as large as vanilla lasso"

# --- Figure 8.2: bar chart of active-coordinate estimates ---
fig, ax = plt.subplots(figsize=(10.0, 4.5))
x_pos = np.arange(10)
width = 0.30
ax.bar(x_pos - width, beta_cv_min[:10], width, color=COLOR_LASSO,
       alpha=0.85, label=r"Vanilla lasso (LassoCV $\lambda_{\min}$)")
ax.bar(x_pos, beta_adapt[:10], width, color=COLOR_DEBIASED,
       alpha=0.85, label=r"Adaptive lasso ($\gamma = 1$, LassoCV init)")
ax.bar(x_pos + width, np.ones(10), width, color=COLOR_TRUTH,
       alpha=0.7, label=r"True $\beta^*_j = 1$")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"$j={j}$" for j in range(10)])
ax.axhline(1.0, color="#aaaaaa", linewidth=0.6, linestyle=":", alpha=0.6)
ax.set_xlabel(r"Active coordinate index $j \in S$")
ax.set_ylabel(r"$\hat\beta_j$")
ax.set_title("Adaptive lasso reduces the constant shrinkage bias on active coordinates")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_08_02_adaptive_vs_vanilla.png", bbox_inches="tight")
plt.show()


## §9. Geometry of the high-dimensional regime

§§5–6 introduced two structural conditions on $\mathbf{X}$ — restricted-eigenvalue (RE) for prediction-risk bounds and irrepresentable (IC) for support recovery — and stated when they hold without unpacking the geometry. This section deepens the picture: the **restricted isometry property** (RIP) from compressed sensing, the implication chain $\mathrm{RIP} \Rightarrow \mathrm{RE}$, sub-Gaussian-design concentration arguments, and when each condition fails in practice.

The section is text-heavy and short — no new theorems beyond §§5–6, just structural unpacking. The payoff is a cleaner picture of what makes a "good" design for the lasso and what doesn't, useful for diagnosing real-data failures.


### §9.1 The restricted isometry property (RIP)

The **restricted isometry property** (Candès-Tao 2005) is a *uniform* version of RE: instead of asking that the design preserve geometry on a particular cone $\mathcal{C}(S, c_0)$, it asks that the design act approximately like an isometry on *all* $s$-sparse vectors.

> **Definition 9.1 (restricted isometry property).** $\mathbf{X}$ satisfies RIP of order $s$ with constant $\delta_s \in (0, 1)$ if for every $\mathbf{v} \in \mathbb{R}^p$ with $\|\mathbf{v}\|_0 \le s$,
> $$
> (1 - \delta_s) \|\mathbf{v}\|_2^2 \;\le\; \tfrac{1}{n} \|\mathbf{X}\mathbf{v}\|_2^2 \;\le\; (1 + \delta_s) \|\mathbf{v}\|_2^2.
> $$

**Why uniform sparsity matters.** RE (§5.4) is keyed to a specific support $S$ and sign $\mathbf{s}^*_S$. RIP, by contrast, holds simultaneously for every $|S| \le s$ — uniformly across the combinatorial $\binom{p}{s}$ possibilities. Much stronger than RE for any single $S$, but also much harder to verify.

**Compressed-sensing origin.** Given a known $\mathbf{X}$ ($n \ll p$) and observed $\mathbf{y} = \mathbf{X}\boldsymbol\beta^*$, recover the unknown sparse $\boldsymbol\beta^*$ via L1 minimization $\hat{\boldsymbol\beta} = \arg\min \|\boldsymbol\beta\|_1$ s.t. $\mathbf{X}\boldsymbol\beta = \mathbf{y}$. This is the lasso in the limit $\lambda \to 0$ with the constraint replaced by exact equality. Candès-Tao (2005): if $\delta_{2s} < \sqrt 2 - 1 \approx 0.414$, the L1 minimizer recovers $\boldsymbol\beta^*$ exactly. The lasso's noisy-version recovery results inherit this geometric content via the RE relaxation.

**When RIP holds.** For random Gaussian designs with iid $\mathcal{N}(0, 1/n)$ entries, RIP of order $s$ with constant $\delta$ holds w.h.p. provided $n \gtrsim s \log(p/s)/\delta^2$. Same scaling for sub-Gaussian random designs (Baraniuk-Davenport-DeVore-Wakin 2008). For *deterministic* designs, RIP is generally hard to verify — checking $\delta_s < c$ requires examining all $\binom{p}{s}$ sparse subsets, computationally intractable.


### §9.2 RIP $\Rightarrow$ RE: the implication chain

> **Proposition 9.1 (RIP $\Rightarrow$ RE).** If $\mathbf{X}$ satisfies RIP of order $2s$ with $\delta_{2s} < 1/(1 + 2c_0)$, then $\mathrm{RE}(s, c_0)$ holds with $\kappa^2 \ge 1 - \delta_{2s}(1 + 2c_0)$.

**Proof sketch.** Pick $\boldsymbol\delta \in \mathcal{C}(S, c_0)$ with $|S| = s$. Decompose $\boldsymbol\delta_{S^c}$ into blocks $T_1, T_2, \dots$ of size $s$ each, sorted by magnitude. Set $T_0 = S$. Each $\boldsymbol\delta_{T_k}$ is $s$-sparse, so RIP applies to it.

The cone condition $\|\boldsymbol\delta_{S^c}\|_1 \le c_0 \|\boldsymbol\delta_S\|_1$, combined with the standard top-$s$ approximation argument, gives $\sum_{k \ge 2} \|\boldsymbol\delta_{T_k}\|_2 \le c_0 \|\boldsymbol\delta_S\|_2$. Triangle inequality on $\mathbf{X}\boldsymbol\delta$:

$$
\tfrac{\|\mathbf{X}\boldsymbol\delta\|}{\sqrt n} \ge \sqrt{1 - \delta_{2s}}\|\boldsymbol\delta_{T_0 \cup T_1}\| - \sqrt{1 + \delta_s} \cdot c_0 \|\boldsymbol\delta_S\|,
$$

squaring gives RE with $\kappa^2 = 1 - \delta_{2s}(1 + 2c_0)$. (The polished version with sharp constants is in Wainwright 2019, §11.2.) $\blacksquare$

**The reverse does not hold.** RE is strictly weaker than RIP. Random Gaussian designs with non-identity covariance — correlated columns — satisfy RE without satisfying RIP. RE for these designs comes from Raskutti-Wainwright-Yu (2010), avoiding RIP entirely.

In practice: "compressed-sensing-style" designs (random Gaussian with iid entries) → RIP framework. "Statistics-style" designs (random with population covariance) → RE framework. Lasso oracle inequality holds under either, but constants differ.


### §9.3 Sub-Gaussian designs and concentration

The §5.5 deviation step controlled $\|\mathbf{X}^\top \boldsymbol\varepsilon / n\|_\infty$ via sub-Gaussian concentration on the *noise*. For random *designs*, a parallel result controls how the empirical Gram $\mathbf{X}^\top \mathbf{X}/n$ approximates the population $\boldsymbol\Sigma$.

> **Lemma 9.1 (sub-Gaussian design concentration; Vershynin 2018, Thm 4.6.1).** Let $\mathbf{X}$ have iid sub-Gaussian rows with mean zero and covariance $\boldsymbol\Sigma$. With probability $\ge 1 - 2\exp(-cn)$,
> $$
> \left\|\tfrac{\mathbf{X}^\top \mathbf{X}}{n} - \boldsymbol\Sigma\right\|_{\text{op}} \le C\left(\sqrt{\tfrac{p}{n}} + \tfrac{p}{n}\right) \|\boldsymbol\Sigma\|_{\text{op}}.
> $$

For $n \gg p$: the empirical Gram concentrates tightly around $\boldsymbol\Sigma$. For $n \asymp p$: concentration is loose — at $n < p$, the smallest eigenvalue of $\mathbf{X}^\top\mathbf{X}/n$ is exactly zero regardless of $\boldsymbol\Sigma$.

For the lasso, we need the *restricted* version on the cone $\mathcal{C}(S, c_0)$:

> **Theorem 9.1 (RE for sub-Gaussian designs; Raskutti-Wainwright-Yu 2010, simplified).** Let $\mathbf{X}$ have iid sub-Gaussian rows with $\lambda_{\min}(\boldsymbol\Sigma) \ge \kappa_0^2 > 0$. If $n \ge C s \log(p)$ for $C$ depending on $\kappa_0$ and the sub-Gaussian parameter, then w.p. $\ge 1 - c_1 \exp(-c_2 n)$, $\mathrm{RE}(s, 3)$ holds with $\kappa^2 \ge \kappa_0^2 / 8$.

The population condition $\lambda_{\min}(\boldsymbol\Sigma) > 0$ + the sample-size scaling $n \ge C s \log(p)$ gives the sample-level RE the lasso needs. This is the standard "high-dimensional probability" path from population assumption to sample-level condition.

For DGP-1 with $\boldsymbol\Sigma_{jk} = 0.5^{|j-k|}$: $\lambda_{\min}(\boldsymbol\Sigma) \to (1-0.5)/(1+0.5) = 1/3$ as $p \to \infty$. The sample-size requirement $n \ge C s \log(p) \approx 60 C$ is satisfied at $n = 200$.


In [ ]:
# §9.3 — Numerical diagnostics on DGP-1: empirical RE constant on the active set,
# plus the population vs empirical RE for comparison.

# Population-level diagnostics (using the AR(1) Toeplitz Sigma).
print(f"Population diagnostics for DGP-1 (Sigma_jk = 0.5^|j-k|, p={X1.shape[1]}):")

idx_full = np.arange(X1.shape[1])
Sigma_pop = 0.5 ** np.abs(np.subtract.outer(idx_full, idx_full))
S_idx = np.arange(10)
Sigma_SS_pop = Sigma_pop[np.ix_(S_idx, S_idx)]
lambda_min_pop      = float(np.linalg.eigvalsh(Sigma_pop).min())
lambda_min_pop_SS   = float(np.linalg.eigvalsh(Sigma_SS_pop).min())
ic_pop_dgp1         = population_ic(X1.shape[1], 10, 0.5, np.ones(10))
print(f"  lambda_min(Sigma)        = {lambda_min_pop:.4f}  (population restricted-eigenvalue lower bound)")
print(f"  lambda_min(Sigma_SS)     = {lambda_min_pop_SS:.4f}  (active-block conditioning)")
print(f"  IC quantity     = {ic_pop_dgp1:.4f}  (lasso sign-consistency requires < 1)")

# Empirical (sample-level) diagnostics on the §1 sample.
print(f"\nEmpirical diagnostics on §1 sample (n={X1.shape[0]}):")

# Empirical Gram matrix and its smallest eigenvalue (= 0 since p > n).
Gram_emp = X1.T @ X1 / X1.shape[0]
ev_min_emp_full = float(np.linalg.eigvalsh(Gram_emp).min())  # ~0
ev_max_emp_full = float(np.linalg.eigvalsh(Gram_emp).max())
print(f"  lambda_min(X^T X / n) = {ev_min_emp_full:.6f}  (~ 0 since p > n)")
print(f"  lambda_max(X^T X / n) = {ev_max_emp_full:.4f}")

# Empirical active-block Gram and its conditioning.
X_S = X1[:, S_idx]
Gram_S = X_S.T @ X_S / X1.shape[0]
ev_S = np.linalg.eigvalsh(Gram_S)
cond_S = float(ev_S.max() / ev_S.min())
print(f"  lambda_min(X_S^T X_S / n) = {ev_S.min():.4f}  (vs population {lambda_min_pop_SS:.4f})")
print(f"  cond(X_S^T X_S / n)  = {cond_S:.2f}     (small => active block well-conditioned)")

# Empirical IC quantity on the §1 sample.
def empirical_ic(X, S_mask, sign_S):
    X_S, X_Sc = X[:, S_mask], X[:, ~S_mask]
    return float(np.max(np.abs(X_Sc.T @ X_S @ np.linalg.solve(X_S.T @ X_S, sign_S))))

S_mask_dgp1 = np.zeros(X1.shape[1], dtype=bool)
S_mask_dgp1[:10] = True
ic_emp_dgp1 = empirical_ic(X1, S_mask_dgp1, np.ones(10))
print(f"  empirical IC quantity = {ic_emp_dgp1:.4f}  (vs population {ic_pop_dgp1:.4f})")

# Sanity: empirical IC should be close to population (within sub-Gaussian deviation).
assert abs(ic_emp_dgp1 - ic_pop_dgp1) < 0.5, \
    "Empirical IC should match population IC up to sub-Gaussian fluctuation"


### §9.4 When the conditions fail in practice

When RE or IC fail, the lasso behaves badly in predictable ways. Three common failure modes:

**Highly correlated features.** When $\rho \to 1$, IC fails first; RE degrades more slowly but eventually fails. Lasso behavior:
- *Support recovery*: lasso may flip arbitrarily between which correlated feature it selects across resamples.
- *Prediction*: surprisingly robust — the lasso's prediction is approximately the same whether it picks feature $j$ or $k$ when $\mathbf{X}_j \approx \mathbf{X}_k$.

Remedies: elastic net (§8.2) for grouped selection; adaptive lasso (§8.3) for sign-consistency under weaker conditions; post-selection refit for unbiased coefficients.

**Deterministic / structured designs.** Vandermonde, Fourier, wavelet bases — RE typically holds only under specific column-sampling protocols. RIP is even harder to verify deterministically. Practical applications usually rely on the design being "random-Gaussian-like" enough that RE holds w.h.p., with empirical diagnostics substituting for rigorous condition checking.

**Adversarial / pathological designs.** Designs where the lasso provably fails: some inactive feature exactly representable as a sign-coherent combination of active features (IC = 1 exactly), or two exactly-equal columns. Don't appear in random data but arise from data preprocessing — one-hot encoding of categorical variables produces columns summing to a constant, violating general position (§2.3). Standard preprocessing — drop one level per one-hot encoding, check for duplicate columns — handles these.

**Diagnostic toolkit.** Practical checks when the lasso behaves unexpectedly:
1. **Max pairwise feature correlation:** if $> 0.9$, consider elastic net.
2. **Condition number of $\mathbf{X}_{\hat A}^\top \mathbf{X}_{\hat A}$:** if $> 100$, the active-set Gram is ill-conditioned and IC is likely violated; consider adaptive lasso or stability selection.
3. **Stability of $\hat A_\lambda$ across resamples:** if the active set varies substantially, IC is likely violated; report stability-selection probabilities (Meinshausen-Bühlmann 2010) instead of a point estimate.
4. **Population-vs-empirical Gram drift:** large $\|\mathbf{X}^\top \mathbf{X}/n - \hat{\boldsymbol\Sigma}\|_{\text{op}}$ on a held-out fold suggests insufficient sample size for stable RE.

None theoretically rigorous; all practically useful.


## §10. Post-selection inference and the debiased lasso

The inferential payoff of the topic. Up to now, the lasso has been a *prediction* tool. When practitioners want *inference* — confidence intervals for individual $\beta^*_j$, hypothesis tests of $\beta^*_j = 0$ — the natural impulse is to treat the lasso fit like an OLS fit and apply standard normal-theory machinery: $\hat\beta_j \pm 1.96 \cdot \widehat{\mathrm{se}}(\hat\beta_j)$.

This **fails dramatically**. The lasso's bias (§4.1) shifts $\hat\beta_j$ away from $\beta^*_j$; the naive standard error doesn't account for selection; the resulting CIs undercover by 20–50 points. Empirical coverage of nominally 95% naive lasso CIs lands at 50–70%.

The fix is the **debiased lasso** (Zhang-Zhang 2014; Javanmard-Montanari 2014; van de Geer et al. 2014): a one-step Newton correction $\hat{\boldsymbol\beta}^{\text{db}} = \hat{\boldsymbol\beta} + (1/n) \hat{\mathbf{M}} \mathbf{X}^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta})$ that explicitly removes the lasso's bias and produces $\sqrt n$-consistent normal estimates of individual coefficients. Works in the $p \gg n$ regime where standard OLS doesn't exist; reduces to OLS at $p < n$.


### §10.1 Why naive post-selection CIs undercover (PoSI)

Standard OLS CIs assume the model is **specified before the data are seen**. The lasso violates this: $\hat A_\lambda$ is selected from the data. Treating the post-lasso refit as if it had been pre-specified **double-counts the data** — once for selection, once for inference — and the resulting CIs are too narrow.

**Berk-Brown-Buja-Zhang-Zhao (2013) — "Valid Post-Selection Inference"** formalized this with **PoSI** intervals: simultaneously valid over all submodels the procedure could have selected. The PoSI multiplier $K_{\text{PoSI}}$ is much larger than $z_{0.025} = 1.96$ — typically $K_{\text{PoSI}} \in [3, 5]$ at $p = 100$, growing as $\sqrt{2\log p}$. PoSI intervals are valid but extremely wide. The debiased lasso takes a fundamentally different approach: rather than widening the CI to absorb selection uncertainty, construct an estimator whose distribution is *not selection-dependent* in the first place.

**Three sources of failure for the naive CI** ("fit lasso, refit OLS on $\hat A_\lambda$, form CI"):
1. **Selection bias on the refit estimator.** OLS on a selected support is biased because the support was chosen to make the regression "look good."
2. **Underestimated standard error.** The OLS variance formula doesn't account for variability in $\hat A$.
3. **Coverage degenerates for noise coords.** For $j \notin \hat A$, the refit doesn't include $\beta_j$ — the implicit estimate is 0, no CI.

Empirical effect at standard SNR: nominally 95% CIs cover at 50–70% of replications.


### §10.2 The one-step debiased correction (Zhang-Zhang 2014)

The debiased lasso starts from a different question: rather than fixing the CI for $\hat{\boldsymbol\beta}^{\text{lasso}}$, construct a *new estimator* whose asymptotic distribution doesn't depend on selection.

The KKT condition (§2.4) gives $\hat{\boldsymbol\Sigma} \hat{\boldsymbol\beta} = \frac{1}{n} \mathbf{X}^\top \mathbf{y} - \lambda \hat{\mathbf{g}}$ where $\hat{\boldsymbol\Sigma} = \mathbf{X}^\top \mathbf{X}/n$ and $\hat{\mathbf{g}} \in \partial \|\hat{\boldsymbol\beta}\|_1$. If $\hat{\boldsymbol\Sigma}^{-1}$ existed: $\hat{\boldsymbol\beta} = \hat{\boldsymbol\Sigma}^{-1} \frac{1}{n}\mathbf{X}^\top\mathbf{y} - \lambda \hat{\boldsymbol\Sigma}^{-1} \hat{\mathbf{g}}$. The first term is OLS; the second is the lasso bias. To remove the bias: add it back. Substituting the KKT condition $\lambda \hat{\mathbf{g}} = \frac{1}{n} \mathbf{X}^\top (\mathbf{y} - \mathbf{X}\hat{\boldsymbol\beta})$:

> **Definition 10.1 (debiased lasso).** Given the lasso initial $\hat{\boldsymbol\beta}$ and a matrix $\hat{\mathbf{M}} \approx \hat{\boldsymbol\Sigma}^{-1}$,
> $$
> \hat{\boldsymbol\beta}^{\text{db}} := \hat{\boldsymbol\beta} + \frac{1}{n} \hat{\mathbf{M}} \mathbf{X}^\top (\mathbf{y} - \mathbf{X} \hat{\boldsymbol\beta}).
> $$

**Three observations:**
1. **At $p < n$ with $\hat{\mathbf{M}} = \hat{\boldsymbol\Sigma}^{-1}$:** $\hat{\boldsymbol\beta}^{\text{db}} = \hat{\boldsymbol\beta}^{\text{OLS}}$ exactly — the debiased lasso reduces to OLS.
2. **At $p > n$:** $\hat{\boldsymbol\Sigma}$ is singular; need to construct $\hat{\mathbf{M}}$ via nodewise lasso (§10.3).
3. **Bias-variance decomposition.** Substituting $\mathbf{y} = \mathbf{X}\boldsymbol\beta^* + \boldsymbol\varepsilon$:
$$
\hat{\boldsymbol\beta}^{\text{db}} - \boldsymbol\beta^* = \underbrace{\tfrac{1}{n}\hat{\mathbf{M}}\mathbf{X}^\top\boldsymbol\varepsilon}_{\text{normal-asymptotic}} + \underbrace{(\hat{\mathbf{M}}\hat{\boldsymbol\Sigma} - \mathbf{I})(\hat{\boldsymbol\beta} - \boldsymbol\beta^*)}_{\text{remainder}}.
$$
The first term is asymptotically normal by CLT. The remainder is $o_p(1/\sqrt n)$ when $\hat{\mathbf{M}}\hat{\boldsymbol\Sigma} \approx \mathbf{I}$ tightly enough.

This is the **one-step Newton** construction: take a biased initial estimator, take a single Newton step toward the unbiased solution. General theory: Le Cam (1956), Bickel (1982); lasso application: Zhang-Zhang (2014).


### §10.3 The nodewise lasso for $\hat{\mathbf{M}}$

In the $p > n$ regime, $\hat{\boldsymbol\Sigma}$ is singular. Target $\hat{\mathbf{M}}$ such that row $j$ of $\hat{\mathbf{M}}$ makes $\hat{\mathbf{M}}_{j, \cdot} \hat{\boldsymbol\Sigma} \approx \mathbf{e}_j$.

**Nodewise lasso (van de Geer-Bühlmann-Ritov-Dezeure 2014, Algorithm 1).** For each $j = 1, \dots, p$:
1. Regress feature $j$ on the others: $\hat{\boldsymbol\gamma}_j := \arg\min_{\boldsymbol\gamma} \frac{1}{2n}\|\mathbf{X}_j - \mathbf{X}_{-j}\boldsymbol\gamma\|^2 + \lambda_j \|\boldsymbol\gamma\|_1$.
2. Compute residual variance: $\hat{\tau}_j^2 = \frac{1}{n}\mathbf{X}_j^\top(\mathbf{X}_j - \mathbf{X}_{-j}\hat{\boldsymbol\gamma}_j)$.
3. Form row $j$ of $\hat{\mathbf{M}}$: $\hat{\mathbf{M}}_{j, k} = 1/\hat\tau_j^2$ if $k = j$, else $-\hat\gamma_{j, k}/\hat\tau_j^2$.

**Motivation: partition formula.** Under population $\boldsymbol\Sigma$, the $j$-th row of $\boldsymbol\Sigma^{-1}$ is $(1, -\boldsymbol\gamma_j^*)/\tau_j^{*2}$ where $\boldsymbol\gamma_j^* = \boldsymbol\Sigma_{-j, -j}^{-1}\boldsymbol\Sigma_{-j, j}$ is the population regression of $X_j$ on $X_{-j}$ and $\tau_j^{*2}$ is the population residual variance. The nodewise lasso estimates $(\boldsymbol\gamma_j^*, \tau_j^{*2})$ in the high-dim regime.

**Computational cost.** $p$ lasso fits — about 30s at $p = 500, n = 200$ with cv=10. Caching $\hat{\mathbf{M}}$ across uses on the same $\mathbf{X}$ amortizes this.

**Sparsity assumption.** Works because rows of $\boldsymbol\Sigma^{-1}$ are assumed sparse — each feature well-predicted by a small subset of others. If $\boldsymbol\Sigma^{-1}$ is dense (every feature depends on many others), $\hat{\mathbf{M}}$ is misspecified and the one-step correction fails.

**Alternatives.** Javanmard-Montanari (2014) construct $\hat{\mathbf{M}}$ via a direct optimization. Belloni-Chernozhukov-Wang (2014) use ridge. Asymptotically equivalent at first order; finite-sample CIs differ by 10–20% in length.


In [ ]:
# §10.3 — Nodewise lasso construction, demonstrated on a single DGP-1 sample.
# Show that the resulting M_hat satisfies M_hat @ Sigma_hat ~ I (where Sigma_hat = X^T X / n).

def nodewise_lasso(X, lam_node=None):
    """Construct M_hat via the nodewise lasso (van de Geer et al. 2014, Algorithm 1).

    For each column j: regress X_j on X_{-j} via lasso, form M_hat_j as the
    "approximate inverse" row.

    Uses a single fixed lambda_j (theory-guided) for all nodes; in production,
    each node would have its own CV-tuned lambda_j.
    """
    n, p = X.shape
    if lam_node is None:
        # Theory-guided: sigma * sqrt(log(p)/n), assuming feature scale ~ 1.
        lam_node = np.sqrt(np.log(p) / n)

    M_hat = np.zeros((p, p))
    for j in range(p):
        # Lasso regression of X_j on X_{-j}.
        X_minus_j = np.delete(X, j, axis=1)
        gamma_j = Lasso(alpha=lam_node, fit_intercept=False,
                        max_iter=10_000, tol=1e-7).fit(X_minus_j, X[:, j]).coef_
        # Residual variance.
        residual = X[:, j] - X_minus_j @ gamma_j
        tau_j_sq = float((X[:, j] @ residual) / n)
        # Build row j of M_hat.
        m_j = np.zeros(p)
        m_j[j] = 1.0
        m_j_others = np.delete(np.arange(p), j)
        m_j[m_j_others] = -gamma_j
        M_hat[j, :] = m_j / tau_j_sq
    return M_hat

# Demonstrate on a smaller sample for a tractable runtime.
n_demo, p_demo = 200, 100
X_demo, _, _, _ = make_dgp1(n=n_demo, p=p_demo, s=5,
                             rng=np.random.default_rng(SEED + 11000))
print(f"Computing nodewise lasso M_hat for n={n_demo}, p={p_demo}...")
M_hat_demo = nodewise_lasso(X_demo)
Sigma_hat_demo = X_demo.T @ X_demo / n_demo

# Verify M_hat @ Sigma_hat ~ I.
product = M_hat_demo @ Sigma_hat_demo
diag_err  = float(np.max(np.abs(np.diag(product) - 1.0)))
offdiag_err = float(np.max(np.abs(product - np.diag(np.diag(product)))))
print(f"  max |diag(M_hat Sigma_hat) - 1|       = {diag_err:.4f}")
print(f"  max |off-diagonal of M_hat Sigma_hat| = {offdiag_err:.4f}")
print(f"  (Both should be small for valid debiased-lasso inference)")

# Sanity: with the theory-guided lambda on a moderate-correlation design, M_hat @ Sigma_hat should
# be close enough to I that the diagonal is within 0.5 of 1 and off-diagonals
# don't dominate.
assert diag_err < 0.5, "Nodewise lasso M_hat should give diag(M_hat Sigma_hat) ~ 1 within tolerance"


### §10.4 The $\sqrt n$ normal asymptotics

> **Theorem 10.1 (asymptotic normality; van de Geer et al. 2014, Theorem 2.2; Javanmard-Montanari 2014, Theorem 2.1).** Under sub-Gaussian iid design with sparse $\boldsymbol\Sigma^{-1}$ (max row sparsity $s_M$), sub-Gaussian noise, $s$-sparse $\boldsymbol\beta^*$, $n \gg \max(s, s_M) (\log p)^2$, and the lasso initial + nodewise $\hat{\mathbf{M}}$ each satisfying the appropriate oracle inequality:
> $$
> \sqrt n (\hat\beta_j^{\text{db}} - \beta^*_j) / \hat\sigma_{db, j} \overset{d}{\to} \mathcal{N}(0, 1),
> $$
> where $\hat\sigma_{db, j}^2 = \sigma^2 \hat{\mathbf{M}}_{j, j} / n$.

**Proof sketch.** From the §10.2 decomposition:
$$
\sqrt n (\hat\beta_j^{\text{db}} - \beta^*_j) = \tfrac{1}{\sqrt n} (\hat{\mathbf{M}}\mathbf{X}^\top \boldsymbol\varepsilon)_j + \sqrt n [(\hat{\mathbf{M}}\hat{\boldsymbol\Sigma} - \mathbf{I})(\hat{\boldsymbol\beta} - \boldsymbol\beta^*)]_j.
$$

*Normal-asymptotic term:* linear in $\boldsymbol\varepsilon$ with deterministic-given-$\mathbf{X}$ coefficients $\Rightarrow$ multivariate CLT $\Rightarrow$ asymptotically $\mathcal{N}(0, \sigma^2 \hat{\mathbf{M}}_{j, j})$.

*Remainder term:* nodewise-lasso KKT bounds $\|(\hat{\mathbf{M}}\hat{\boldsymbol\Sigma} - \mathbf{I})_{j, \cdot}\|_\infty \le \lambda_j$. Combine with §5.5 lasso L1 rate $\|\hat{\boldsymbol\beta} - \boldsymbol\beta^*\|_1 = O_p(s\sqrt{\log(p)/n})$. The remainder is $O_p(\lambda_j \cdot s\sqrt{\log(p)/n}) = O_p(s\log(p)/n) = o_p(1/\sqrt n)$ when $n \gg s^2 (\log p)^2$. The slightly weaker theorem requirement $n \gg s_M (\log p)^2$ uses a tighter analysis. $\blacksquare$

**The $(\log p)^2$ scaling.** The debiased lasso requires substantially more samples than the lasso's prediction-risk bound (which only needed $n \gg s\log p$). The extra $\log p$ comes from $\hat{\mathbf{M}}$ requiring its own oracle inequality. In practice the debiased lasso works at smaller $n$ than the theory requires.

**Confidence interval.** Asymptotically valid $(1 - \alpha)$ CI for $\beta^*_j$:
$$
\mathrm{CI}_{db, j}^{1-\alpha} = \hat\beta_j^{\text{db}} \pm z_{\alpha/2} \, \hat\sigma \sqrt{\hat{\mathbf{M}}_{j, j} / n}.
$$
Use $\hat\sigma^2 = \|\mathbf{y} - \mathbf{X}\hat{\boldsymbol\beta}\|^2 / (n - \|\hat{\boldsymbol\beta}\|_0)$ for the noise variance estimate (Reid-Tibshirani-Friedman 2016).

**Hypothesis test.** For $H_0: \beta^*_j = 0$: reject at level $\alpha$ if $|\hat\beta_j^{\text{db}}| > z_{\alpha/2} \hat\sigma \sqrt{\hat{\mathbf{M}}_{j, j}/n}$. Correct asymptotic level regardless of whether the lasso selected $j$.


### §10.5 Empirical coverage demonstration

We compare three CI procedures on DGP-1-style data at $(n, p, s) = (200, 100, 5)$ — the user-specified setting where OLS is feasible as a baseline:

1. **OLS CI** (gold standard): standard normal-theory CI from OLS coefficients/standard errors.
2. **Naive post-selection CI**: refit OLS on $\hat A_\lambda$, use refit standard error.
3. **Debiased lasso CI**: Zhang-Zhang one-step with $\hat{\mathbf{M}} = \hat{\boldsymbol\Sigma}^{-1}$ (the OLS Hessian inverse, available since $p < n$).

For each method and each coordinate, form a nominally 95% CI; check coverage of the true $\beta^*_j$ across $B = 200$ MC replicates. Coverage reported separately for **signal coords** ($j \in S$, $\beta^*_j = 1$) and **noise coords** ($j \notin S$, $\beta^*_j = 0$).

**Expected pattern (Figure 10.1):**
- **OLS CI** covers ~95% at both — gold standard.
- **Naive post-selection CI** undercovers at signal coords (~50–70%) due to selection-induced bias.
- **Debiased lasso CI** covers ~95% at both — recovers gold-standard coverage from a biased lasso initial.

The headline takeaway: **the debiased lasso recovers OLS-quality inference from a biased lasso initial**. At $p > n$ where OLS is unavailable, the debiased lasso (with nodewise $\hat{\mathbf{M}}$) is the only valid CI procedure of the three.


In [ ]:
# §10.5 — Empirical coverage Monte Carlo with MIXED signal strengths.
# At (n=200, p=100), use an active set with both strong and borderline signals
# so that the lasso's selected support actually fluctuates across MC reps.
# Otherwise (cf. uniform beta*=1, sigma=0.5 setting) selection is essentially
# deterministic, A_hat == S almost always, and naive post-selection CIs look
# fine — masking the conditional-coverage problem this section is designed to
# expose.
#
# Mixed signal regime:
#   coords 0-2:   beta*=1.0  (strong cluster, almost-always selected)
#   coords 50-51: beta*=0.15 (borderline, selected ~40% of reps)
#   everything else: beta*=0.0 (noise)
#
# Borderline coords sit ~50 indices away from the strong cluster so that the
# AR(1) cross-correlation rho^|j-k| ~ rho^48 ~ 0 — the borderline signals are
# effectively independent of the strong block, and their selection probability
# is governed by their own marginal SNR rather than leakage from neighbors.
# β*=0.15 was chosen by sweeping the selection probability: at sigma=0.5,
# rho=0.5, n=200, p=100, the lasso at theory-guided lambda selects coord 50
# in ~37% of reps — the regime where naive post-selection CIs collapse.

n_cov, p_cov = 200, 100
sigma_cov = 0.5
B_cov = 200

beta_star_mixed = np.zeros(p_cov)
beta_star_mixed[:3]    = 1.0    # strong signals
beta_star_mixed[50:52] = 0.15   # borderline signals (selection prob ~ 0.4)

# Coordinates to track, partitioned into three groups for the comparison.
strong_coords     = list(range(3))                  # j = 0, 1, 2
borderline_coords = [50, 51]                         # genuinely borderline
noise_coords      = [j for j in range(10, 100)
                      if j not in borderline_coords]  # exclude borderline

def make_dgp_mixed(n, p, beta_star, sigma, rho, rng):
    """DGP-1 design with caller-specified beta_star (mixed signal strengths)."""
    idx = np.arange(p)
    Sigma = rho ** np.abs(np.subtract.outer(idx, idx))
    L = np.linalg.cholesky(Sigma)
    Z = rng.standard_normal((n, p))
    X = Z @ L.T
    eps = sigma * rng.standard_normal(n)
    y = X @ beta_star + eps
    return X, y

# Theory-guided lambda for the lasso initial estimator.
lam_cov = 2 * sigma_cov * np.sqrt(2 * np.log(p_cov) / n_cov)

# Storage: covered[method][coord_type][rep] = True if the CI covered beta*_j.
methods_list = ["OLS", "Naive post-sel", "Debiased lasso"]
coverage = {m: {"strong": [], "borderline": [], "noise": []}
            for m in methods_list}

z_975 = 1.96  # 95% CI quantile

for b in range(B_cov):
    rng_b = np.random.default_rng(SEED + 12000 + b)
    X_b, y_b = make_dgp_mixed(n_cov, p_cov, beta_star_mixed,
                               sigma=sigma_cov, rho=0.5, rng=rng_b)

    # OLS estimates and standard errors.
    beta_ols = np.linalg.lstsq(X_b, y_b, rcond=None)[0]
    resid_ols = y_b - X_b @ beta_ols
    sigma2_ols = float((resid_ols @ resid_ols) / (n_cov - p_cov))
    Sigma_inv = np.linalg.inv(X_b.T @ X_b)               # (X^T X)^{-1}, NOT scaled by n
    se_ols = np.sqrt(sigma2_ols * np.diag(Sigma_inv))    # OLS standard errors

    # Lasso fit at theory-guided lambda.
    fit_lasso = Lasso(alpha=lam_cov, fit_intercept=False,
                       max_iter=10_000, tol=1e-7).fit(X_b, y_b)
    beta_lasso = fit_lasso.coef_
    active = np.abs(beta_lasso) > 1e-8

    # Naive post-selection: refit OLS on selected support.
    naive_lower = np.full(p_cov, np.nan)
    naive_upper = np.full(p_cov, np.nan)
    naive_point = np.zeros(p_cov)
    if active.sum() >= 1 and active.sum() < n_cov:
        X_active = X_b[:, active]
        beta_refit = np.linalg.lstsq(X_active, y_b, rcond=None)[0]
        resid_refit = y_b - X_active @ beta_refit
        df = n_cov - active.sum()
        sigma2_refit = float((resid_refit @ resid_refit) / df)
        Sigma_active_inv = np.linalg.inv(X_active.T @ X_active)
        se_active = np.sqrt(sigma2_refit * np.diag(Sigma_active_inv))
        active_indices = np.where(active)[0]
        for k, j in enumerate(active_indices):
            naive_point[j] = beta_refit[k]
            naive_lower[j] = beta_refit[k] - z_975 * se_active[k]
            naive_upper[j] = beta_refit[k] + z_975 * se_active[k]
    # For j not in A_hat: naive CI is degenerate {0} — implicit point estimate 0.
    # Cover iff beta*_j == 0.

    # Debiased lasso: beta_db = beta_hat + (1/n) M_hat X^T (y - X beta_hat),
    # with M_hat = (X^T X / n)^{-1}.
    M_hat_cov = np.linalg.inv(X_b.T @ X_b / n_cov)
    beta_db = beta_lasso + (1.0 / n_cov) * M_hat_cov @ X_b.T @ (y_b - X_b @ beta_lasso)
    sigma2_db = float(np.sum((y_b - X_b @ beta_lasso)**2)
                       / (n_cov - max(int(active.sum()), 1)))
    se_db = np.sqrt(sigma2_db * np.diag(M_hat_cov) / n_cov)

    # Track coverage at each method x coord_type.
    for coord_type, coord_list in [("strong", strong_coords),
                                    ("borderline", borderline_coords),
                                    ("noise", noise_coords)]:
        for j in coord_list:
            beta_true_j = beta_star_mixed[j]
            # OLS.
            covered_ols = (beta_ols[j] - z_975 * se_ols[j] <= beta_true_j <=
                            beta_ols[j] + z_975 * se_ols[j])
            coverage["OLS"][coord_type].append(covered_ols)
            # Naive: degenerate when j not in A_hat.
            if active[j]:
                covered_naive = (naive_lower[j] <= beta_true_j <= naive_upper[j])
            else:
                covered_naive = (beta_true_j == 0.0)
            coverage["Naive post-sel"][coord_type].append(covered_naive)
            # Debiased.
            covered_db = (beta_db[j] - z_975 * se_db[j] <= beta_true_j <=
                           beta_db[j] + z_975 * se_db[j])
            coverage["Debiased lasso"][coord_type].append(covered_db)

# Compute coverage rates.
print(f"\nEmpirical coverage rates (B={B_cov} MC reps, nominal 95% CIs):")
print(f"{'Method':<22}  {'Strong (b*=1.0)':>17}  {'Borderline (b*=0.15)':>22}  {'Noise (b*=0)':>14}")
for m in methods_list:
    rate_strong = float(np.mean(coverage[m]["strong"]))
    rate_border = float(np.mean(coverage[m]["borderline"]))
    rate_noise  = float(np.mean(coverage[m]["noise"]))
    print(f"{m:<22}  {rate_strong:>17.3f}  {rate_border:>21.3f}  {rate_noise:>14.3f}")

# Sanity assertions — borderline coords are where naive CIs break down.
ols_strong_cov   = float(np.mean(coverage["OLS"]["strong"]))
ols_border_cov   = float(np.mean(coverage["OLS"]["borderline"]))
naive_border_cov = float(np.mean(coverage["Naive post-sel"]["borderline"]))
db_border_cov    = float(np.mean(coverage["Debiased lasso"]["borderline"]))

assert ols_strong_cov   > 0.85, "OLS strong-signal coverage should be near 95%"
assert ols_border_cov   > 0.85, "OLS borderline coverage should be near 95% (no selection involved)"
assert db_border_cov    > 0.80, "Debiased lasso borderline coverage should remain near 95%"
assert naive_border_cov < ols_border_cov - 0.10, \
    "Naive post-selection CI should undercover BORDERLINE signals by >= 10 pts vs OLS"

# --- Figure 10.1 — Coverage bar chart, three coord types ---
fig, ax = plt.subplots(figsize=(10.5, 5.0))
x_pos = np.arange(3)
width = 0.27
strong_rates = [float(np.mean(coverage[m]["strong"]))     for m in methods_list]
border_rates = [float(np.mean(coverage[m]["borderline"])) for m in methods_list]
noise_rates  = [float(np.mean(coverage[m]["noise"]))      for m in methods_list]

bars1 = ax.bar(x_pos - width, strong_rates, width, color=COLOR_TRUTH,
               alpha=0.85, label=r"Strong signal ($\beta^*_j = 1.0$, almost always selected)")
bars2 = ax.bar(x_pos,         border_rates, width, color=COLOR_LASSO,
               alpha=0.85, label=r"Borderline signal ($\beta^*_j = 0.15$, sometimes selected)")
bars3 = ax.bar(x_pos + width, noise_rates,  width, color=COLOR_OLS,
               alpha=0.85, label=r"Noise ($\beta^*_j = 0$)")
ax.axhline(0.95, color=COLOR_DEBIASED, linewidth=1.5, linestyle="--",
           label="Nominal 95%")

# Annotate bars.
for bars, rates in [(bars1, strong_rates), (bars2, border_rates), (bars3, noise_rates)]:
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, rate + 0.015,
                f"{rate:.2f}", ha="center", fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(methods_list, fontsize=10)
ax.set_ylabel("Empirical coverage")
ax.set_ylim(0, 1.12)
ax.set_title(f"Confidence-interval coverage with MIXED signal strengths "
             f"($n = {n_cov}, p = {p_cov}$; 3 strong + 2 borderline + 88 noise; $B = {B_cov}$ reps)\n"
             "Naive post-selection CIs collapse at borderline coords --- selection bias in action.")
ax.legend(loc="lower left", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_10_01_debiased_coverage.png", bbox_inches="tight")
plt.show()


## §11. Generalized lasso for non-Gaussian responses

Everything in §§1–10 was developed for the **Gaussian linear model**: $\mathbf{y} = \mathbf{X}\boldsymbol\beta^* + \boldsymbol\varepsilon$ with squared-error loss and soft-thresholding KKT solutions. The lasso framework extends naturally to **generalized linear models** (GLMs) — replace the squared-error loss with any GLM negative log-likelihood. Most results from §§5, 6, 10 carry over with family-specific constants.

This section covers two extensions (logistic for binary, Poisson for counts), the general IRLS-with-soft-thresholding solver, and the GLM-debiased-lasso construction for inference. The math is mostly mechanical; we sketch the changes without re-deriving the proofs.


### §11.1 Logistic lasso for binary classification

For binary $y_i \in \{0, 1\}$ with $\mathbb{P}(y_i = 1 \mid \mathbf{x}_i) = \sigma(\mathbf{x}_i^\top \boldsymbol\beta)$ where $\sigma(z) = 1/(1 + e^{-z})$, the negative log-likelihood per observation is

$$
\ell(y_i, \mathbf{x}_i^\top \boldsymbol\beta) = -y_i \mathbf{x}_i^\top \boldsymbol\beta + \log(1 + \exp(\mathbf{x}_i^\top \boldsymbol\beta)).
$$

The **logistic lasso**: $\hat{\boldsymbol\beta}(\lambda) = \arg\min \frac{1}{n} \sum_i \ell(y_i, \mathbf{x}_i^\top \boldsymbol\beta) + \lambda \|\boldsymbol\beta\|_1$. Convex.

**Two structural differences from the Gaussian case:**

1. **No closed-form, even on orthogonal designs.** Each coordinate update involves a non-trivial root-finding step.
2. **The Hessian is data-dependent.** $\nabla^2 \ell = \mathbf{X}^\top \mathbf{W}(\boldsymbol\beta) \mathbf{X}/n$ with $\mathbf{W}(\boldsymbol\beta) = \text{diag}(\sigma(\mathbf{x}_i^\top \boldsymbol\beta)(1 - \sigma(\mathbf{x}_i^\top \boldsymbol\beta)))$ — a $\boldsymbol\beta$-dependent diagonal weighting. Conditioning is state-dependent: when many predicted probabilities are near 0 or 1, $\mathbf{W}$ has small entries and the Hessian is poorly conditioned.

**Solver: cyclic coordinate descent with IRLS quadratic approximations.** At each outer iteration, form a weighted-Gaussian-lasso quadratic approximation around the current $\boldsymbol\beta^{(t)}$, solve via §3.2 coordinate descent, update. Iterate. `glmnet`'s logistic variant is the reference. scikit-learn: `LogisticRegression(penalty='l1', solver='saga')` for general use, `solver='liblinear'` for small problems.

**Theory carries over.** Logistic-lasso analog of the §5 oracle inequality: under RE on the *weighted* design $\mathbf{W}^{1/2}(\boldsymbol\beta^*)\mathbf{X}$, prediction risk in deviance bounded by $C\sigma^2 s \log(p)/n$ for appropriate $\sigma$ (van de Geer 2008; Bühlmann-van de Geer 2011 §6). Constants messier; rate identical.


In [ ]:
# §11.1 — Logistic lasso fit on a binary version of the §1 DGP.
# Generate continuous X from DGP-1, transform y to a binary response via the
# logistic link, fit LogisticRegression(penalty='l1').

# Use the §1 sample's X.
X_logit = X1
n_logit, p_logit = X_logit.shape
beta_star_logit = beta_star_1.copy()

# Generate binary y from the logistic model: P(y_i = 1) = sigma(X_i^T beta*).
linear_pred = X_logit @ beta_star_logit
prob_y1 = 1.0 / (1.0 + np.exp(-linear_pred))
y_binary = (np.random.default_rng(SEED + 13000).uniform(size=n_logit) < prob_y1).astype(int)
print(f"Logistic DGP: n={n_logit}, p={p_logit}, true s=10, "
      f"P(y=1) range = [{prob_y1.min():.3f}, {prob_y1.max():.3f}], "
      f"empirical P(y=1) = {y_binary.mean():.3f}")

# Fit logistic lasso. sklearn's LogisticRegression uses C = 1/(n*lambda) parameterization.
lam_logit = 0.1
C_logit = 1.0 / (n_logit * lam_logit)
fit_logit = LogisticRegression(l1_ratio=1.0, solver='saga', C=C_logit,
                                fit_intercept=False, max_iter=10_000, tol=1e-4).fit(X_logit, y_binary)
beta_logit = fit_logit.coef_.flatten()

# Diagnostics.
n_active_logit = int(np.sum(np.abs(beta_logit) > 1e-8))
n_in_S_logit = int(np.sum(np.abs(beta_logit[:10]) > 1e-8))
mean_active_logit = float(np.mean(np.abs(beta_logit[:10])))
print(f"\nLogistic lasso (sklearn liblinear, C = 1/(n*lambda) with lambda = {lam_logit}):")
print(f"  active set size:               {n_active_logit}")
print(f"  active coords in S = {{0..9}}:    {n_in_S_logit}/10")
print(f"  mean |beta_j| on S:              {mean_active_logit:.4f}  "
      f"(true beta*_j = 1; logistic shrinks more than Gaussian)")

# Sanity: should recover most of the true active set.
assert n_in_S_logit >= 6, \
    "Logistic lasso should recover at least 6 of 10 true active coords"


### §11.2 Poisson lasso for count data

For count $y_i \in \{0, 1, 2, \dots\}$ with $y_i \mid \mathbf{x}_i \sim \mathrm{Poisson}(\exp(\mathbf{x}_i^\top \boldsymbol\beta))$, the negative log-likelihood is

$$
\ell(y_i, \mathbf{x}_i^\top \boldsymbol\beta) = -y_i \mathbf{x}_i^\top \boldsymbol\beta + \exp(\mathbf{x}_i^\top \boldsymbol\beta) + \text{const}.
$$

The **Poisson lasso** has the same form as the logistic case. Convex; same IRLS-with-coordinate-descent solver pattern; no closed form. Hessian weighting $\mathbf{W}(\boldsymbol\beta) = \text{diag}(\exp(\mathbf{x}_i^\top \boldsymbol\beta))$ — the predicted Poisson means.

**Applications.** GWAS rare-variant counts, web traffic prediction, text n-gram counts, insurance claim counts — any sparse rate-modeling problem.

**Overdispersion caveat.** Real count data often has $\mathbb{V}\mathrm{ar}(y_i) > \mathbb{E}[y_i]$. The negative-binomial lasso handles this with an overdispersion parameter; `statsmodels.NegativeBinomial(penalty='elastic_net')` is the standard implementation.

scikit-learn provides `PoissonRegressor` for unpenalized Poisson regression; for the L1-penalized version, `glmnet` (via Python bindings) is the reference. Pure-NumPy IRLS implementation is straightforward but rarely needed.


### §11.3 Inference extensions for GLMs

The §10 debiased-lasso construction generalizes from squared-error to any GLM negative log-likelihood. Key change: $\hat{\mathbf{M}}$ now approximates the inverse of the **GLM Hessian** $\nabla^2 \ell(\hat{\boldsymbol\beta}) = \mathbf{X}^\top \mathbf{W}(\hat{\boldsymbol\beta}) \mathbf{X} / n$, not the unweighted Gram $\mathbf{X}^\top \mathbf{X} / n$.

**The GLM-debiased lasso.** Given the lasso initial $\hat{\boldsymbol\beta}^{\text{GLM-lasso}}$ and $\hat{\mathbf{M}}$ approximating the inverse GLM Hessian:

$$
\hat{\boldsymbol\beta}^{\text{GLM-db}} := \hat{\boldsymbol\beta}^{\text{GLM-lasso}} + \frac{1}{n} \hat{\mathbf{M}} \mathbf{X}^\top (\mathbf{y} - g^{-1}(\mathbf{X} \hat{\boldsymbol\beta}^{\text{GLM-lasso}})),
$$

with $g^{-1}$ the GLM inverse-link function ($\sigma$ for logistic, $\exp$ for Poisson). Same one-step Newton interpretation as Gaussian.

**Nodewise lasso for $\hat{\mathbf{M}}$ in GLMs.** *Weighted* nodewise lasso: regress each $\mathbf{X}_j$ on $\mathbf{X}_{-j}$ with sample weights $w_i = (\mathbf{W}(\hat{\boldsymbol\beta}))_{ii}$ from the GLM Hessian. Resulting $\hat{\mathbf{M}}$ approximates $(\mathbf{X}^\top \mathbf{W}(\hat{\boldsymbol\beta}) \mathbf{X}/n)^{-1}$ row-by-row.

**Asymptotic normality (van de Geer et al. 2014, Theorem 3.1).** Under analogous conditions to Theorem 10.1 — sub-Gaussian design, sparse population GLM-Hessian inverse, $s$-sparse $\boldsymbol\beta^*$, $n \gg \max(s, s_M)(\log p)^2$ — the GLM-debiased lasso satisfies $\sqrt n(\hat\beta_j^{\text{GLM-db}} - \beta^*_j) \overset{d}{\to} \mathcal{N}(0, \mathrm{var}_j)$ with explicit $\mathrm{var}_j$. CIs and hypothesis tests follow the §10.4 recipe.

**Practical note.** R's `hdi` package implements the GLM-debiased lasso for logistic and Poisson. Python implementations are less mature; for applications, fit via `LogisticRegression(penalty='l1')` then use R's `hdi` for CIs.

**The bigger picture.** The L1 penalty is a regularizer, not a likelihood-specific construction. It composes with any convex log-likelihood — Gaussian (squared error), Bernoulli (logistic), Poisson (count), Cox (survival), multinomial (multiclass) — giving a family of sparse penalized estimators. Same structural flavor (sparsity, bias, IC for selection, RE for prediction, debiased inference) with family-specific details. Friedman-Hastie-Tibshirani (2010, "Regularization Paths for Generalized Linear Models via Coordinate Descent") is the canonical computational reference; Bühlmann-van de Geer (2011) the statistical reference.


## §12. Connections, applications, and limits

The closing section. The lasso is one of the most widely-used algorithms in modern statistics — standalone estimator, nuisance estimator inside larger inferential pipelines, conceptual ancestor of an entire family of penalized methods. The §§5–10 results transfer with appropriate modifications to most descendants.

This section traces five connections:
- §12.1 Double / debiased ML — the lasso as nuisance estimator.
- §12.2 Causal inference with high-dim confounders.
- §12.3 Sparse-Bayesian alternatives.
- §12.4 Where the lasso breaks down.
- §12.5 Forward pointers in formalML.


### §12.1 Double / debiased ML (Chernozhukov et al. 2018)

The §10 debiased lasso targets inference on individual coefficients of a high-dim linear regression. **Double / debiased machine learning** (DML; Chernozhukov-Chetverikov-Demirer-Duflo-Hansen-Newey-Robins 2018) generalizes this to inference on a low-dim *target parameter* $\theta$ in a model where the nuisance is high-dimensional and estimable by any sufficiently regular ML method.

**The setup.** Target $\theta_0$ identified by $\mathbb{E}[\psi(W; \theta_0, \eta_0)] = 0$ where $W$ is observed data, $\psi$ is a known score, and $\eta_0$ is high-dim nuisance. Naive plug-in $\frac{1}{n}\sum_i \psi(W_i; \hat\theta, \hat\eta) = 0$ is biased — regularization in $\hat\eta$ propagates into $\hat\theta$, giving slower-than-$\sqrt n$ rate.

**Two ingredients fix this:**

1. **Neyman orthogonality:** $\partial_\eta \mathbb{E}[\psi(W; \theta_0, \eta_0)] = 0$ at the truth. The moment condition is *insensitive to first-order errors in the nuisance*. Construct via influence-function correction.
2. **Cross-fitting:** $K$-fold split; for $i$ in fold $k$, use $\hat\eta^{(-k)}$ trained on data minus fold $k$. Decouples $\hat\eta$ from the residual structure used in the moment condition, eliminating the in-sample overfitting bias.

**Theorem (DML asymptotic normality).** Under Neyman orthogonality, cross-fitting, and $\|\hat\eta^{(-k)} - \eta_0\|_{L^2} = o_p(n^{-1/4})$, the DML estimator satisfies $\sqrt n(\hat\theta^{\text{DML}} - \theta_0) \overset{d}{\to} \mathcal{N}(0, \text{var})$ with consistently estimable variance.

**The lasso's role.** Any ML method achieving $o_p(n^{-1/4})$ works. The lasso is the standard choice for sparse high-dim nuisance — the §5 oracle inequality gives $O_p(\sqrt{s\log(p)/n})$, beating $n^{-1/4}$ when $s\log p/n \to 0$. Random forests, boosting, neural networks are alternatives.

**Connection to debiased lasso (§10).** The §10 debiased lasso is "DML for the special case of inference on an individual regression coefficient." The one-step Newton correction follows from applying DML influence-function machinery to $\mathbb{E}[X_j(Y - X^\top \beta_0)] = 0$. Both rely on Neyman orthogonal scores; both achieve $\sqrt n$ asymptotic normality despite regularized nuisance.


### §12.2 High-dim confounder adjustment in causal inference

The most consequential DML application: causal inference with many confounders. Setup: observational data $\{(D_i, X_i, Y_i)\}_{i=1}^n$ with binary treatment $D_i$, high-dim confounders $X_i$, outcome $Y_i$. Goal: average treatment effect $\tau = \mathbb{E}[Y_i(1) - Y_i(0)]$.

Identification (unconfoundedness $Y(d) \perp D \mid X$, overlap $0 < \mathbb{P}(D = 1 \mid X) < 1$) gives the **AIPW representation**:

$$
\tau = \mathbb{E}\left[\mu(X, 1) - \mu(X, 0) + \frac{D - \pi(X)}{\pi(X)(1 - \pi(X))}(Y - \mu(X, D))\right],
$$

where $\pi(x) = \mathbb{P}(D = 1 \mid X = x)$ is the **propensity score** and $\mu(x, d) = \mathbb{E}[Y \mid X = x, D = d]$ is the **outcome regression**. Both nuisance functions are high-dim regression problems estimable by lasso (logistic for $\pi$, Gaussian for $\mu$).

**Doubly robust** (Robins-Rotnitzky-Zhao 1994): the moment condition is satisfied if either $\hat\pi$ or $\hat\mu$ is consistent (not both required). DML strengthens this: under $o_p(n^{-1/4})$ rates for both (achievable by lasso under sparsity), the cross-fit DML estimator $\hat\tau^{\text{DML}}$ is $\sqrt n$-consistent and asymptotically normal with the semiparametrically efficient variance.

**Practical pipeline:** (i) Fit $\hat\pi$ via cross-fitted `LogisticRegression(penalty='l1')`. (ii) Fit $\hat\mu(\cdot, 0)$ and $\hat\mu(\cdot, 1)$ via cross-fitted Lasso. (iii) Plug into AIPW. (iv) Standard error from empirical influence function. (v) CI / test.

Implemented in `econml` (Microsoft Research), `doubleml` (Python), `DoubleML` (R). Standard observational-causal-inference workflow. Forward pointer: T6 `causal-inference-methods`.


### §12.3 Sparse-Bayesian alternatives ([Sparse Bayesian Priors](/topics/sparse-bayesian-priors))

The lasso has a **Bayesian interpretation**: it's the posterior mode (MAP) under an iid Laplace prior $\beta_j \sim \mathrm{Laplace}(0, 1/\lambda)$. The posterior *mean* is dense — Bayesian Laplace inference doesn't produce sparse point estimates, only sparse modes.

**The Bayesian counterpart of the lasso** is a class of priors producing **approximate sparsity** in the posterior — most posterior mass near zero, heavy tails for active coefficients. Two main families (covered in [Sparse Bayesian Priors](/topics/sparse-bayesian-priors)):

- **Horseshoe** (Carvalho-Polson-Scott 2010). $\beta_j \sim \mathcal{N}(0, \lambda_j^2 \tau^2)$ with $\lambda_j \sim C^+(0, 1)$ and $\tau \sim C^+(0, 1)$. Half-Cauchy local scales: pole at zero (heavy shrinkage of small signals), polynomial tails (vanishing shrinkage of large signals) — the inverse of the lasso's constant shrinkage. The active-coefficient bias from §4.1 is avoided.
- **Spike-and-slab** (Mitchell-Beauchamp 1988; George-McCulloch 1993). $\beta_j \sim (1-w)\delta_0 + w\mathcal{N}(0, \sigma^2)$ — discrete mixture of a point mass at zero (spike) and a wide Gaussian (slab). Posterior gives natural variable-selection probabilities $\hat w_j$.

**Trade-offs.** Bayesian methods give native uncertainty quantification — credible intervals from the posterior, no debiased construction needed. Cost: HMC / NUTS sampling is 100×–1000× slower than the lasso. Choice depends on whether $n, p$ make MCMC tractable.


### §12.4 Where the lasso breaks down

Five failure modes beyond the §6.2 / §9.4 IC and RE issues:

**Highly correlated features** (recap from §6.1, §9.4). Lasso flips arbitrarily; elastic net is the remedy.

**Ultra-high-dim regimes ($p > 10^6$).** Coordinate descent's $O(np)$ per-iteration cost becomes prohibitive. **Sure independence screening** (Fan-Lv 2008): preprocess by ranking $|\mathbf{X}_j^\top \mathbf{y}|$ to filter $p$ to $O(n)$ candidates, then run lasso. Loses some precision but keeps things tractable. Standard for genuine $p \gg n^{10}$ genomics regimes.

**Heavy-tailed (non-sub-Gaussian) noise.** §5.5 deviation step requires sub-Gaussian $\boldsymbol\varepsilon$. With heavier tails (Cauchy, low-df Student-$t$), $\|\mathbf{X}^\top\boldsymbol\varepsilon/n\|_\infty \le \lambda/2$ fails at the standard $\lambda$. **Quantile / median lasso** (Belloni-Chernozhukov 2011): replace squared-error with check loss $\rho_\tau(u) = u(\tau - \mathbb{1}\{u < 0\})$. Same $\sqrt{s\log p/n}$ rate under weaker noise assumptions; needs a different solver (LP or smoothed CD).

**Time-series / spatially correlated observations.** Lasso assumes iid; correlated observations bias CV / BIC tuning (nearby train and test points share information). **Block-CV** (block-wise leave-one-out with contiguous blocks) is the standard remedy. Theory under dependence is looser; Wong-Li-Tewari (2020) for stationary time series.

**Causal interpretation pitfalls.** Frequent misuse: interpreting lasso coefficients as "causal effects." The lasso minimizes *prediction* error; coefficients are predictively useful but not causally interpretable without structural assumptions. The §12.1–§12.2 DML / AIPW pipeline is the right framework when causal interpretation is the goal — the lasso's role is *nuisance*, not causal coefficient producer.


### §12.5 Forward pointers in formalML

The lasso machinery feeds into several planned formalML topics:

- **`semiparametric-inference`** (T6, planned). DML framework introduced in §12.1 generalizes — any target $\theta_0$ identified by a moment condition with high-dim nuisance can be estimated $\sqrt n$-consistently using cross-fitted ML nuisance. Lasso is one of several admissible nuisance estimators.

- **`causal-inference-methods`** (T6, planned). Treatment-effect inference with high-dim confounders (§12.2) is the headline DML application. Will cover identification assumptions, AIPW, double-robust efficient scores, the DML estimator and its CI / tests, sensitivity analysis, dynamic treatments, IV connection.

- **`pac-bayes-bounds`** (T6, planned). Catoni's (2007) PAC-Bayesian framework gives an alternative theoretical perspective on sparse regression — a PAC-Bayes prior with a sparsity-favoring structure derives a closely-related risk bound. Connects to the §12.3 Bayesian sparsity framework.

- **`bayesian-neural-networks`** (T5, planned). Sparse priors on neural-net weights — particularly horseshoe priors on input-layer weights — give automatic feature selection in deep models. The §12.3 framework scaled up to neural-net parameter counts.

- **`density-ratio-estimation`** (T5, planned). Estimating $r(x) = p_2(x)/p_1(x)$ — KLIEP and sparse-regularized variants reduce to convex optimization problems structurally similar to the lasso.

**Closing thesis.** The lasso is one of the most successful algorithms in modern statistics because it occupies a sweet spot in the bias-variance-computability triangle. Sparsity bias is small under reasonable conditions; sparsity-adapted variance is the lowest of any L1-style estimator; the convex-optimization formulation makes computation tractable at scales where every pre-2000 alternative was infeasible. The §5 oracle inequality and the §10 debiased-lasso construction together make the lasso the standard tool for both *prediction* and *inference* in the sparse high-dim regime, with adaptations (elastic net, adaptive lasso, GLM-lasso, DML-lasso) covering the cases where the basic version doesn't fit. We've worked through the full pipeline; the rest of the formalML topic graph picks up the threads from here.
